# EDA - Data Understanding (v3)
```text
eda_data_understanding_v3.py
============================
Customer Health & Sentiment Scoring Multi-Agent AI Solution
EDA v2 = original eda_data_understanding.py (logic unchanged) + review points from "EDA for dataset.docx".

Run    : python eda_data_understanding_v3.py              # runs every section top to bottom
         python eda_data_understanding_v3.py --rebuild    # force rebuild of cm / cm_feat tables
         VS Code: each "# %%" is a cell -> "Run Cell" / "Run All" in the Interactive window (charts shown inline)
Low RAM: set EDA_MEMORY=3GB and EDA_THREADS=2 as environment variables before running
Needs  : pandas, duckdb, matplotlib, numpy, scipy, scikit-learn (IPython optional)

Changes to existing code (only where necessary):
  1. charts: shown inline when run interactively; in a plain run they are saved and closed exactly as before
  2. first/second-clause analyses labelled QA ONLY - not modelling features (*_QA_ONLY.csv)
  3. cohort selection scorecard moved from Step 11 to Step 12 (outputs s12_*)

New sections: 1A config | 3A category mapping check | 4A customer-month features & feedback aggregation |
              7A CSAT/NPS missingness & response propensity | 7B CSAT/NPS feature checks |
              11 Behavioral Health Score calibration (+ separate Sentiment and Loyalty scores)

v3 additions (existing logic unchanged):
  3A   : interaction channel mapped to CHANNEL_GROUPS (+ unmapped-channel check) and 3A.1 channel analysis (Q1-Q5)
  7C   : interaction channel x type / sentiment / resolution / CSAT & NPS / FCR (only comparisons not already in Step 7)
  7D   : interaction type comparison + technical_issue / feedback / product_request / compliment profiles
  7E   : engagement channel x events / polarity / conversion / unsubscribe / session duration + push vs quality
  7F   : cross-dataset: interaction vs engagement channel, negative sentiment / complaints / CSAT vs engagement

---------------------------------------------------------------------------------------------------
Original header:
eda_data_understanding.py
=========================
Customer Health & Sentiment Scoring Multi-Agent AI Solution
Data understanding (EDA): Steps 1-10 + cohort selection (Part B)

Stack  : Python + DuckDB (VS Code)
Input  : data/bronze/{customers,transactions,engagements,interactions}.parquet  (ZSTD)
Output : reports/eda/   -> one CSV per check, PNG charts, eda_summary.md (auto-flagged findings)
Period : 1 Oct 2025 - 30 Sep 2026, grain = customer_id x month

Run    : python eda_data_understanding.py              # all steps
         python eda_data_understanding.py 1 2 7        # only selected steps
         python eda_data_understanding.py 11           # step 11 = cohort selection
         python eda_data_understanding.py --rebuild    # force rebuild of customer-month table
Low RAM: set EDA_MEMORY=3GB and EDA_THREADS=2 as environment variables before running
Note   : steps 5-11 use a customer-month table (cm) built once and cached in reports/eda/eda_work.duckdb
```

In [ ]:
import os
import sys
import time
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib

try:                                  # Jupyter / VS Code Interactive -> charts shown inline
    get_ipython()                     # noqa: F821  (defined only inside IPython)
    IN_NOTEBOOK = True
    get_ipython().run_line_magic("matplotlib", "inline")   # noqa: F821  v3: render charts inline in Jupyter
except NameError:                     # plain `python` run -> save charts only (original behaviour)
    IN_NOTEBOOK = False
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# CONFIG  (change here only)
# =============================================================================
BRONZE_DIR = Path("data/bronze")
OUT_DIR = Path("reports/eda")
WORK_DB = OUT_DIR / "eda_work.duckdb"          # stores the customer-month table for re-runs

WINDOW_START = "2025-10-01"                    # inclusive
WINDOW_END = "2026-10-01"                      # exclusive (data ends 30 Sep 2026)

MIN_HISTORY_MONTHS = 3        # months observed (incl. current) before a customer-month is scored
CHURN_GAP_MONTHS = 3          # inactive >= N months since last activity -> churn proxy
RARE_CAT_PCT = 1.0            # category share below this % = rare
MOM_ALERT_PCT = 30.0          # month-on-month change above this % = flag
SEGMENT_SPREAD_ALERT = 1.30   # max/min segment mean above this = segment-relative scoring needed
HIGH_CORR = 0.80              # |spearman| above this = redundant features
TEXT_SAMPLE_PER_LABEL = 34    # ~100 rows for manual reading
CORR_SAMPLE_ROWS = 500_000
MIN_COHORT_CUSTOMERS = 20_000 # sample-size gate for a candidate cohort
SEED = 42
DUCKDB_THREADS = int(os.getenv("EDA_THREADS", 8))
DUCKDB_MEMORY = os.getenv("EDA_MEMORY", "8GB")       # lower it if your laptop has less RAM

SPEND_TYPES = ("purchase", "subscription_payment", "bill_payment")   # gross spend = completed only
COMPLAINT_TYPES = ("complaint", "billing_dispute", "cancellation_request")
OPEN_RESOLUTION = ("unresolved", "escalated", "pending")
FEE_TYPES = ("fee",)  # fee transactions tracked separately from gross spend

TABLES = {  # table -> primary key
    "customers": "customer_id",
    "transactions": "transaction_id",
    "engagements": "engagement_id",
    "interactions": "interaction_id",
}

# Cohort definitions (SQL on table cm)
HISTORY_COND = f"active AND months_observed >= {MIN_HISTORY_MONTHS}"
CORE_COND = f"{HISTORY_COND} AND has_tx AND (has_eng OR has_int)"

### 1A. v2 configuration (NEW - business mappings and scoring settings)
The mappings below are verified against the data in Section 3A before any feature uses them.
Edit them here if the frequency report shows a mismatch.

In [ ]:
import gc
import numpy as np
try:
    from IPython.display import display, Markdown
except ImportError:                   # plain Python without IPython installed
    def display(obj):
        print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

    def Markdown(text):
        return text
from scipy.stats import spearmanr, chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SILVER_DIR = Path("data/silver")
GOLD_DIR = Path("data/gold")

# ---- business mappings (verified in Section 3A) ----
PAYMENT_TYPES = ("subscription_payment", "bill_payment")
REFUND_TYPES = ("refund",)
TX_SUCCESS, TX_FAILED, TX_REVERSED = "completed", "failed", "reversed"
RESOLVED_STATUS = ("resolved",)
ENG_POSITIVE = ("login", "feature_use", "offer_redeem", "email_click", "push_open", "sms_click", "sms_reply")
ENG_NEUTRAL = ("page_view", "offer_view", "email_open", "session_end")
ENG_NEGATIVE = ("unsubscribe", "sms_optout", "push_dismiss")      # more events is NOT always healthier
UNSUBSCRIBE_EVENTS = ("unsubscribe", "sms_optout")
STRICT_MAPPING = False        # True = stop the run if a mapping does not match the data

# ---- survey scales (valid ratings only; invalid values are flagged and excluded, never repaired) ----
CSAT_MIN, CSAT_MAX = 1, 5
NPS_MIN, NPS_MAX = 0, 10
NPS_LOYALTY_POINTS = {"detractor": 0, "passive": 50, "promoter": 100}   # category-based loyalty score

# ---- Step 7A ----
PROPENSITY_SAMPLE = 300_000
MCAR_AUC_ALERT = 0.55         # propensity AUC above this -> survey response is predictable from observables

# ---- Step 11: Behavioral Health Score calibration ----
BHS_WEIGHTS = {               # initial component weights (sum = 1); only OBSERVED components are used, then re-normalised
    "tx_reliability": 0.15, "spend_level": 0.15, "spend_trend": 0.10,
    "engagement_level": 0.10, "engagement_quality": 0.10, "engagement_trend": 0.10,
    "service_experience": 0.15, "csat_health": 0.15,
}
INCLUDE_CSAT_IN_BHS = True    # CSAT is a BHS component (validated leave-CSAT-out); NPS stays OUT of BHS (independent check)
TREAT_MISSING_TX_AS_ZERO = False   # no transaction record != zero transactions (source may be incomplete)
MIN_BHS_WEIGHT_COVERAGE = 0.30     # minimum share of base weight observed to publish a BHS
TEXT_SENTIMENT_PATH = SILVER_DIR / "text_sentiment_predictions.parquet"   # Sentiment Agent output: interaction_id, text_sentiment_pred
CAL_END = "2026-04-01"        # in-time (calibration) months < CAL_END <= out-of-time months
OUTCOME_HORIZON = 3           # months ahead for future outcomes
SENS_SAMPLE = 300_000
SENS_RANDOM_RUNS = 50
OOT_SAMPLE = 500_000          # customer-months sampled for out-of-time metrics (keeps memory low)

# ---- v3: interaction channel business mapping (verified in Section 3A) ----
CHANNEL_GROUPS = {                     # group -> interaction channels
    "assisted_live": ("phone", "branch"),                # real-time, agent-led
    "assisted_digital": ("chat", "in_app_support"),      # real-time, digital, agent-led
    "asynchronous": ("email",),                          # delayed response
    "public_social": ("social_media",),                  # public / reputational risk
}
CHANNEL_MATERIAL_LIFT = 1.10    # channel rate / overall rate >= this -> materially higher (flagged)
CHANNEL_MIN_N = 500             # minimum interactions for a channel to be ranked
CHANNEL_EFFECT_MIN = 0.02       # Cramer's V / epsilon-squared below this = negligible difference (large n makes p-values ~0)

# ---- v3: Step 7D interaction-type profiles (metrics requested per type) ----
TYPE_PROFILES = {
    "technical_issue": ["pct_negative", "pct_positive", "pct_neutral", "avg_csat", "resolution_rate", "fcr_rate", "unresolved_rate"],
    "feedback": ["pct_positive", "pct_negative", "pct_neutral", "avg_csat", "avg_nps", "nps_index"],
    "product_request": ["pct_positive", "pct_neutral", "pct_negative", "avg_csat",
                        "pct_resolved", "pct_escalated", "pct_pending", "pct_unresolved"],
    "compliment": ["pct_positive", "avg_csat", "avg_nps", "nps_index"],
}

# ---- v3: Step 7E push-notification analysis ----
PUSH_CHANNEL = "push_notification"
PUSH_OPEN_EVENTS = ("push_open",)
PUSH_DISMISS_EVENTS = ("push_dismiss",)

## 2. Helpers

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(WORK_DB))
con.execute(f"SET threads={DUCKDB_THREADS}")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}'")
con.execute(f"SET temp_directory='{(OUT_DIR / 'duckdb_tmp').as_posix()}'")   # spill to disk instead of crashing
con.execute("SET preserve_insertion_order=false")
FINDINGS = []


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


def sql_list(values) -> str:
    return "(" + ", ".join(f"'{v}'" for v in values) + ")"


def yes(col: str) -> str:
    return f"(upper(trim(CAST({col} AS VARCHAR))) IN ('Y','YES','TRUE','1'))"


def header(title: str):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)


def save(df: pd.DataFrame, name: str, show: int = 10) -> pd.DataFrame:
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"\n--- {name}  ({len(df):,} rows) -> {path}")
    if show:
        with pd.option_context("display.max_columns", 60, "display.width", 250,
                               "display.float_format", "{:,.3f}".format):
            print(df.head(show).to_string(index=False))
    return df


def flag(step: str, msg: str):
    FINDINGS.append((step, msg))
    print(f"  [FLAG] {msg}")


def columns_of(view: str) -> list:
    return q(f"DESCRIBE {view}")["column_name"].tolist()


def view_exists(name: str) -> bool:
    return bool(q(f"SELECT count(*) n FROM information_schema.tables WHERE table_name = '{name}'").iloc[0, 0])

In [ ]:
# v2 helpers (NEW)
for d in (SILVER_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)


def run_step(fn):
    t0 = time.time()
    fn()
    print(f"\n(done in {time.time() - t0:,.1f}s)")


def cramers_v(ct: pd.DataFrame):
    chi2, p, _, _ = chi2_contingency(ct)
    k = min(ct.shape) - 1
    return (float(np.sqrt(chi2 / (ct.values.sum() * k))) if k > 0 else 0.0), float(p)


def auc_safe(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    m = ~np.isnan(s)
    return roc_auc_score(y[m], s[m]) if m.sum() > 10 and len(np.unique(y[m])) == 2 else np.nan

In [ ]:
def finish_plot():
    """Show the chart when running interactively; close it in a plain run (original behaviour). PNG is always saved."""
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close()

## 3. Sources: raw views + typed views

In [ ]:
def register_sources():
    for t in TABLES:
        file_ = BRONZE_DIR / f"{t}.parquet"
        folder = BRONZE_DIR / t
        if file_.exists():
            src = f"read_parquet('{file_.as_posix()}')"
        elif folder.exists():
            src = f"read_parquet('{(folder / '**' / '*.parquet').as_posix()}')"
        else:
            raise FileNotFoundError(f"Parquet for '{t}' not found in {BRONZE_DIR}")
        con.execute(f"CREATE OR REPLACE VIEW raw_{t} AS SELECT * FROM {src}")

    win = f"TIMESTAMP '{WINDOW_START}'", f"TIMESTAMP '{WINDOW_END}'"

    con.execute("""
        CREATE OR REPLACE VIEW cust AS
        SELECT *, TRY_CAST(signup_date AS DATE) AS signup,
               TRY_CAST(age AS DOUBLE) AS age_n, TRY_CAST(tenure_months AS DOUBLE) AS tenure_n
        FROM raw_customers
    """)
    con.execute("""
        CREATE OR REPLACE VIEW cust1 AS            -- one row per customer (guards against duplicates)
        SELECT * FROM cust
        QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY signup NULLS LAST) = 1
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW tx AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(transaction_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(amount AS DOUBLE) AS amt, TRY_CAST(discount_amount AS DOUBLE) AS disc
              FROM raw_transactions)
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW eng AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(event_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(session_duration_sec AS DOUBLE) AS dur,
                     TRY_CAST(pages_viewed AS DOUBLE) AS pages,
                     {yes('is_converted')} AS converted
              FROM raw_engagements)
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW itx AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(interaction_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(duration_sec AS DOUBLE) AS dur,
                     TRY_CAST(csat_score AS DOUBLE) AS csat,
                     TRY_CAST(nps_score AS DOUBLE) AS nps,
                     TRY_CAST(sentiment_score_gt AS DOUBLE) AS sscore,
                     CAST(interaction_text AS VARCHAR) AS txt,
                     {yes('first_contact_resolution')} AS fcr
              FROM raw_interactions)
    """)


EVENT_VIEWS = {"transactions": "tx", "engagements": "eng", "interactions": "itx"}
TS_COL = {"transactions": "transaction_ts", "engagements": "event_ts", "interactions": "interaction_ts"}


register_sources()

## 3A. Category frequency report and business-mapping verification (NEW)
Hard-coded lists (SPEND_TYPES, COMPLAINT_TYPES, OPEN_RESOLUTION, payment / refund / engagement groups) are checked
against the actual category distributions before any feature uses them:
* every configured value must exist in the data (catches typos and renamed categories);
* every observed value is listed with the group it is mapped to, so unmapped categories are visible;
* every engagement event must be classified as positive / neutral / negative.

In [ ]:
def step3a_category_mapping():
    header("SECTION 3A - Category frequencies and business-mapping verification")
    columns = [("tx", "transaction_type"), ("tx", "transaction_status"), ("itx", "interaction_type"),
               ("itx", "resolution_status"), ("itx", "channel"), ("itx", "sentiment_label"), ("eng", "event_type"),
               ("eng", "engagement_channel")]
    freq = []
    for v, c in columns:
        d = q(f"""SELECT CAST({c} AS VARCHAR) value, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
                  FROM {v} WHERE in_window GROUP BY 1 ORDER BY n DESC""")
        freq.append(d.assign(source=v, column=c))
    freq = pd.concat(freq)[["source", "column", "value", "n", "pct"]]

    mappings = {   # (source, column) -> {group: values}
        ("tx", "transaction_type"): {"SPEND_TYPES": SPEND_TYPES, "FEE_TYPES": FEE_TYPES, "PAYMENT_TYPES": PAYMENT_TYPES, "REFUND_TYPES": REFUND_TYPES},
        ("tx", "transaction_status"): {"TX_SUCCESS": (TX_SUCCESS,), "TX_FAILED": (TX_FAILED,), "TX_REVERSED": (TX_REVERSED,)},
        ("itx", "interaction_type"): {"COMPLAINT_TYPES": COMPLAINT_TYPES},
        ("itx", "resolution_status"): {"OPEN_RESOLUTION": OPEN_RESOLUTION, "RESOLVED_STATUS": RESOLVED_STATUS},
        ("eng", "event_type"): {"ENG_POSITIVE": ENG_POSITIVE, "ENG_NEUTRAL": ENG_NEUTRAL, "ENG_NEGATIVE": ENG_NEGATIVE},
        ("itx", "channel"): {f"CHANNEL_{g.upper()}": vals for g, vals in CHANNEL_GROUPS.items()},     # v3
    }

    def groups_of(src, col, val):
        return ", ".join(g for g, vals in mappings.get((src, col), {}).items() if val in vals) or "-- not mapped --"

    freq["mapped_to"] = [groups_of(s, c, v) for s, c, v in zip(freq.source, freq.column, freq.value)]
    save(freq, "s3a_category_frequency_and_mapping", show=60)

    issues = []
    for (src, col), groups in mappings.items():
        observed = set(freq[(freq.source == src) & (freq.column == col)].value)
        for g, vals in groups.items():
            for v in vals:
                if v not in observed:
                    issues.append({"mapping": g, "column": col, "value": v, "issue": "configured value NOT found in data"})
    for v in freq[(freq.column == "event_type") & (freq.mapped_to == "-- not mapped --")].value:
        issues.append({"mapping": "ENG_*", "column": "event_type", "value": v, "issue": "event not mapped to positive/neutral/negative"})
    for v in freq[(freq.column == "channel") & (freq.mapped_to == "-- not mapped --")].value:     # v3
        issues.append({"mapping": "CHANNEL_*", "column": "channel", "value": v, "issue": "interaction channel not mapped to a channel group"})
    for v in set(UNSUBSCRIBE_EVENTS) - set(ENG_NEGATIVE):
        issues.append({"mapping": "UNSUBSCRIBE_EVENTS", "column": "event_type", "value": v, "issue": "unsubscribe event not in ENG_NEGATIVE"})
    issues = save(pd.DataFrame(issues, columns=["mapping", "column", "value", "issue"]), "s3a_mapping_issues", show=20)
    if len(issues):
        flag("S3A", f"{len(issues)} business-mapping issue(s) - review s3a_mapping_issues.csv before trusting derived features")
        if STRICT_MAPPING:
            raise ValueError("Business mapping does not match the data (STRICT_MAPPING=True)")
    else:
        flag("S3A", "All configured business mappings exist in the data and every engagement event is classified")

    cov = pd.DataFrame([{
        "pct_tx_spend_types": q(f"SELECT round(100.0*count_if(transaction_type IN {sql_list(SPEND_TYPES)})/count(*), 2) FROM tx WHERE in_window").iloc[0, 0],
        "pct_tx_fee_types": q(f"SELECT round(100.0*count_if(transaction_type IN {sql_list(FEE_TYPES)})/count(*), 2) FROM tx WHERE in_window").iloc[0, 0],
        "pct_tx_success": q(f"SELECT round(100.0*count_if(transaction_status = '{TX_SUCCESS}')/count(*), 2) FROM tx WHERE in_window").iloc[0, 0],
        "pct_int_complaint": q(f"SELECT round(100.0*count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)})/count(*), 2) FROM itx WHERE in_window").iloc[0, 0],
        "pct_int_open": q(f"SELECT round(100.0*count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)})/count(*), 2) FROM itx WHERE in_window").iloc[0, 0],
        "pct_eng_negative": q(f"SELECT round(100.0*count_if(event_type IN {sql_list(ENG_NEGATIVE)})/count(*), 2) FROM eng WHERE in_window").iloc[0, 0]}])
    save(cov, "s3a_mapping_coverage")


run_step(step3a_category_mapping)

### 3A.1 Interaction channel mapping and analysis (NEW - v3)
Every interaction channel is mapped to a business channel group (CHANNEL_GROUPS in 1A) and the mapping is checked above.
This sub-step answers five channel questions; each answer is compared with the all-channel rate (lift) and an effect
size, because with ~1M interactions almost any difference is "statistically significant".

| # | Question | Metric |
|---|---|---|
| Q1 | Which channel has the highest complaint rate? | complaint / billing dispute / cancellation (COMPLAINT_TYPES) share |
| Q2 | Which channel has the highest proportion of negative sentiment? | sentiment_label = negative share (label = EDA only) |
| Q3 | Which channel has the highest unresolved interaction rate? | OPEN_RESOLUTION share |
| Q4 | Does CSAT vary across channels? | mean / median valid CSAT, Kruskal-Wallis + epsilon-squared |
| Q5 | Are cancellation requests more frequent through specific channels? | cancellation share, lift vs overall, chi-square + Cramer's V |

In [ ]:
def step3a_channel_analysis():
    header("SECTION 3A.1 - Interaction channel mapping and channel analysis")
    from scipy.stats import kruskal
    ch_group = "CASE " + " ".join(f"WHEN channel IN {sql_list(v)} THEN '{g}'" for g, v in CHANNEL_GROUPS.items()) + " ELSE 'unmapped' END"
    csat_ok = f"CASE WHEN csat BETWEEN {CSAT_MIN} AND {CSAT_MAX} AND csat = round(csat) THEN csat END"
    base_sql = f"""
        SELECT CAST(channel AS VARCHAR) channel, {ch_group} channel_group, count(*) n,
               count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)}) n_complaint,
               count_if(sentiment_label = 'negative') n_negative,
               count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)}) n_unresolved,
               count_if(interaction_type = 'cancellation_request') n_cancel,
               count({csat_ok}) n_csat, avg({csat_ok}) avg_csat, median({csat_ok}) median_csat,
               100.0 * count_if({csat_ok} <= 2) / nullif(count({csat_ok}), 0) pct_low_csat
        FROM itx WHERE in_window GROUP BY ALL"""
    k = q(base_sql)
    tot = k[["n", "n_complaint", "n_negative", "n_unresolved", "n_cancel"]].sum()
    for m, col in (("complaint_rate", "n_complaint"), ("negative_pct", "n_negative"),
                   ("unresolved_rate", "n_unresolved"), ("cancellation_rate", "n_cancel")):
        k[m] = 100 * k[col] / k.n
        k[m.replace("_rate", "").replace("_pct", "") + "_lift"] = k[m] / (100 * tot[col] / tot.n)
    k["cancellation_share_of_all_cancellations"] = 100 * k.n_cancel / tot.n_cancel
    k["interaction_share"] = 100 * k.n / tot.n
    k = save(k.sort_values("n", ascending=False).round(3), "s3a_channel_kpis", show=10)
    grp = k.groupby("channel_group")[["n", "n_complaint", "n_negative", "n_unresolved", "n_cancel"]].sum().reset_index()
    for m, col in (("complaint_rate", "n_complaint"), ("negative_pct", "n_negative"),
                   ("unresolved_rate", "n_unresolved"), ("cancellation_rate", "n_cancel")):
        grp[m] = (100 * grp[col] / grp.n).round(3)
    save(grp, "s3a_channel_group_kpis")

    ranked = k[k.n >= CHANNEL_MIN_N]
    answers = []
    for qid, question, rate, cnt in (
            ("Q1", "Highest complaint rate", "complaint_rate", "n_complaint"),
            ("Q2", "Highest proportion of negative sentiment", "negative_pct", "n_negative"),
            ("Q3", "Highest unresolved interaction rate", "unresolved_rate", "n_unresolved"),
            ("Q5", "Cancellation requests more frequent through specific channels", "cancellation_rate", "n_cancel")):
        top = ranked.sort_values(rate, ascending=False).iloc[0]
        overall = 100 * tot[cnt] / tot.n
        v, p = cramers_v(pd.DataFrame({"yes": ranked[cnt], "no": ranked.n - ranked[cnt]}))
        material = (top[rate] / overall >= CHANNEL_MATERIAL_LIFT) and v >= CHANNEL_EFFECT_MIN
        answers.append({"question": qid, "topic": question, "top_channel": top.channel, "top_channel_group": top.channel_group,
                        "top_value_pct": round(top[rate], 2), "overall_pct": round(overall, 2),
                        "lift_vs_overall": round(top[rate] / overall, 3),
                        "spread_pp": round(ranked[rate].max() - ranked[rate].min(), 2),
                        "effect_size": round(v, 4), "effect_metric": "Cramer's V", "p_value": p,
                        "conclusion": "materially higher" if material else "highest, but differences across channels are negligible"})

    # Q4 - CSAT across channels (valid ratings only, no imputation)
    cs = q(f"SELECT CAST(channel AS VARCHAR) channel, {csat_ok} csat FROM itx WHERE in_window AND {csat_ok} IS NOT NULL")
    groups = [g.csat.to_numpy() for _, g in cs.groupby("channel") if len(g) >= CHANNEL_MIN_N]
    h, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
    n_all, n_grp = sum(len(g) for g in groups), len(groups)
    eps2 = (h - n_grp + 1) / (n_all - n_grp) if n_all > n_grp else np.nan      # epsilon-squared effect size
    ranked_c = ranked.dropna(subset=["avg_csat"])
    lo, hi = ranked_c.sort_values("avg_csat").iloc[0], ranked_c.sort_values("avg_csat").iloc[-1]
    answers.append({"question": "Q4", "topic": "Does CSAT vary across channels?", "top_channel": f"high: {hi.channel} / low: {lo.channel}",
                    "top_channel_group": f"{hi.channel_group} / {lo.channel_group}",
                    "top_value_pct": round(hi.avg_csat, 3), "overall_pct": round(float(cs.csat.mean()), 3),
                    "lift_vs_overall": round(hi.avg_csat / lo.avg_csat, 3), "spread_pp": round(hi.avg_csat - lo.avg_csat, 3),
                    "effect_size": round(float(eps2), 5), "effect_metric": "epsilon-squared (Kruskal-Wallis)", "p_value": p,
                    "conclusion": "CSAT varies materially by channel" if eps2 >= CHANNEL_EFFECT_MIN
                                  else "CSAT is effectively the same across channels"})
    ans = save(pd.DataFrame(answers).sort_values("question"), "s3a_channel_questions", show=10)
    for r in ans.itertuples():
        flag("S3A", f"{r.question} {r.topic}: {r.top_channel} ({r.top_value_pct} vs overall {r.overall_pct}; "
                    f"{r.effect_metric} {r.effect_size}) -> {r.conclusion}")

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    x = np.arange(len(ranked)); w = 0.2
    for i, (m, lab) in enumerate((("complaint_rate", "complaint %"), ("negative_pct", "negative sentiment %"),
                                  ("unresolved_rate", "unresolved %"), ("cancellation_rate", "cancellation %"))):
        ax[0].bar(x + (i - 1.5) * w, ranked[m], w, label=lab)
    ax[0].set_xticks(x); ax[0].set_xticklabels(ranked.channel, rotation=30, ha="right"); ax[0].legend(fontsize=8)
    ax[0].set_title("Channel risk rates (% of channel interactions)")
    ax[1].bar(ranked_c.channel, ranked_c.avg_csat, color="#4472C4"); ax[1].set_ylim(CSAT_MIN, CSAT_MAX)
    ax[1].tick_params(axis="x", rotation=30); ax[1].set_title("Average valid CSAT by channel")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s3a_channel_analysis.png", dpi=120); finish_plot()


run_step(step3a_channel_analysis)

## 4. Customer-month table cm (used by Steps 5-12)
REBUILD_CM = True (or `--rebuild` on the command line) forces a rebuild, same as the original script.

In [ ]:
def build_cm(force: bool = False):
    if view_exists("cm") and not force:
        print("cm table already exists in work DB (use --rebuild to refresh)")
        return
    t0 = time.time()
    print("Building customer-month table cm (customers x 12 months) ...")
    con.execute(f"""
    CREATE OR REPLACE TABLE cm AS
    WITH months AS (
        SELECT CAST(range AS DATE) AS mth
        FROM range(TIMESTAMP '{WINDOW_START}', TIMESTAMP '{WINDOW_END}', INTERVAL 1 MONTH)
    ),
    spine AS (
        SELECT c.customer_id, m.mth,
               (c.signup IS NOT NULL AND m.mth < date_trunc('month', c.signup)) AS pre_signup
        FROM cust1 c CROSS JOIN months m
    ),
    tx_m AS (
        SELECT
            customer_id,
            mth,
            -- Overall transaction counts
            COUNT(*) AS n_tx,
            COUNT_IF(transaction_status = 'completed') AS n_tx_completed,
            COUNT_IF(transaction_status = 'failed') AS n_tx_failed,
            COUNT_IF(transaction_status = 'reversed') AS n_tx_reversed,
            -- Refund transaction count
            COUNT_IF(transaction_type = 'refund') AS n_refunds,
            -- Gross spending: completed purchases and payments only
            COALESCE(SUM(amt) FILTER (
                WHERE transaction_status = 'completed'
                AND transaction_type IN {sql_list(SPEND_TYPES)}
            ), 0) AS gross_spend,
            -- Fee transaction counts across all statuses
            COUNT_IF(transaction_type IN {sql_list(FEE_TYPES)}) AS n_fees,
            COUNT_IF(transaction_type IN {sql_list(FEE_TYPES)} AND transaction_status = 'completed') AS n_fees_completed,
            COUNT_IF(transaction_type IN {sql_list(FEE_TYPES)} AND transaction_status = 'failed') AS n_fees_failed,
            COUNT_IF(transaction_type IN {sql_list(FEE_TYPES)} AND transaction_status = 'reversed') AS n_fees_reversed,
            -- Positive fee amounts from completed transactions
            COALESCE(SUM(amt) FILTER (
                WHERE transaction_type IN {sql_list(FEE_TYPES)}
                AND transaction_status = 'completed'
            ), 0) AS fee_completed_amt,
            -- Positive fee amounts from reversed transactions
            COALESCE(SUM(ABS(amt)) FILTER (
                WHERE transaction_type IN {sql_list(FEE_TYPES)}
                AND transaction_status = 'reversed'
            ), 0) AS fee_reversed_amt,
            -- Net fee amount after accounting for reversals
            COALESCE(SUM(amt) FILTER (
                WHERE transaction_type IN {sql_list(FEE_TYPES)}
                AND transaction_status = 'completed'
            ), 0)
            -
            COALESCE(SUM(ABS(amt)) FILTER (
                WHERE transaction_type IN {sql_list(FEE_TYPES)}
                AND transaction_status = 'reversed'
            ), 0) AS net_fee_amt,
            -- Refund amount
            COALESCE(SUM(ABS(amt)) FILTER (
                WHERE transaction_status = 'completed'
                AND transaction_type = 'refund'
            ), 0) AS refund_amt,
            -- Other transaction information
            COALESCE(SUM(disc), 0) AS discount_amt,
            MODE(currency) AS tx_currency
        FROM tx
        WHERE in_window
        GROUP BY ALL
    ),
    eng_m AS (
        SELECT customer_id, mth,
            count(*) AS n_eng,
            count_if(event_type = 'login') AS n_logins,
            count_if(event_type = 'offer_redeem') AS n_redeem,
            count_if(converted) AS n_conversions,
            coalesce(sum(dur), 0) AS eng_seconds,
            coalesce(sum(pages), 0) AS pages_viewed,
            count(DISTINCT engagement_channel) AS n_eng_channels
        FROM eng WHERE in_window GROUP BY ALL
    ),
    itx_m AS (
        SELECT customer_id, mth,
            count(*) AS n_int,
            count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)}) AS n_complaint,
            count_if(interaction_type = 'cancellation_request') AS n_cancel_req,
            count_if(sentiment_label = 'negative') AS n_negative,
            count_if(sentiment_label = 'positive') AS n_positive,
            count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)}) AS n_open_issue,
            count_if(fcr) AS n_fcr,
            avg(sscore) AS avg_sentiment_gt,
            avg(csat) AS avg_csat, count(csat) AS n_csat,
            avg(nps) AS avg_nps, count(nps) AS n_nps
        FROM itx WHERE in_window GROUP BY ALL
    ),
    base AS (
        SELECT s.customer_id, s.mth, s.pre_signup,
               (t.customer_id IS NOT NULL) AS has_tx,
               (e.customer_id IS NOT NULL) AS has_eng,
               (i.customer_id IS NOT NULL) AS has_int,
               t.* EXCLUDE (customer_id, mth),
               e.* EXCLUDE (customer_id, mth),
               i.* EXCLUDE (customer_id, mth)
        FROM spine s
        LEFT JOIN tx_m t ON s.customer_id = t.customer_id AND s.mth = t.mth
        LEFT JOIN eng_m e ON s.customer_id = e.customer_id AND s.mth = e.mth
        LEFT JOIN itx_m i ON s.customer_id = i.customer_id AND s.mth = i.mth
    ),
    lc AS (
        SELECT *,
            CAST(has_tx AS INT) + CAST(has_eng AS INT) + CAST(has_int AS INT) AS n_sources,
            (has_tx OR has_eng OR has_int) AS active
        FROM base
    ),
    lc2 AS (
        SELECT *,
            MIN(CASE WHEN active THEN mth END) OVER (PARTITION BY customer_id) AS first_active_mth,
            MAX(CASE WHEN active THEN mth END) OVER w_prev AS last_active_before,
            SUM(CAST(active AS INT)) OVER w_cum AS active_months_to_date,
            LAG(active) OVER w_ord AS active_prev
        FROM lc
        WINDOW w_ord AS (PARTITION BY customer_id ORDER BY mth),
               w_cum AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
               w_prev AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    )
    SELECT *,
        CASE WHEN first_active_mth IS NOT NULL AND mth >= first_active_mth
             THEN date_diff('month', first_active_mth, mth) + 1 ELSE 0 END AS months_observed,
        date_diff('month', last_active_before, mth) AS months_since_last_active,
        CASE WHEN active AND mth = first_active_mth THEN 'new_in_data'
             WHEN active AND active_prev THEN 'retained'
             WHEN active THEN 'reactivated'
             WHEN last_active_before IS NULL THEN 'not_yet_active'
             WHEN date_diff('month', last_active_before, mth) < {CHURN_GAP_MONTHS} THEN 'dormant'
             ELSE 'churn_proxy' END AS lifecycle_status,
        COALESCE(NULLIF(CONCAT_WS('+', CASE WHEN has_tx THEN 'TX' END,
                                       CASE WHEN has_eng THEN 'ENG' END,
                                       CASE WHEN has_int THEN 'INT' END), ''), 'none') AS source_combo,
        CASE WHEN has_tx THEN coalesce(gross_spend, 0) + coalesce(net_fee_amt, 0) END AS total_customer_outflow,
        CASE WHEN has_tx THEN PERCENT_RANK() OVER (PARTITION BY mth, tx_currency ORDER BY gross_spend) END AS spend_pctile
    FROM lc2
    """)
    n = q("SELECT COUNT(*) FROM cm").iloc[0, 0]
    print(f"cm built: {n:,} rows in {time.time() - t0:,.1f}s")


REBUILD_CM = False
build_cm(force=REBUILD_CM)

## 4A. Extended customer-month features and feedback aggregation (NEW - the original cm table is unchanged)
| Table | Grain | Content |
|---|---|---|
| fb_i -> data/silver/fact_feedback_interaction.parquet | interaction | valid CSAT/NPS, csat_normalized (0-100), NPS category & loyalty points, invalid flags |
| fb_m | customer x month | feedback aggregation: CSAT & NPS counts, averages, latest rating, category counts |
| cm_feat -> data/silver/customer_month_features.parquet | customer x month (active) | customers, interactions, CSAT, NPS, engagements, transactions, trends, source-presence flags |

Rules: CSAT/NPS counts use valid ratings only (invalid values flagged and excluded); survey values stay NULL when not
observed (no imputation); has_transactions = 0 means "no record", not "zero transactions"; positive vs negative engagement
are separated; payment failures, refunds, reversals and successes are separated; amounts stay in their own currency.

In [ ]:
def build_cm_features(force: bool = False):
    if view_exists("cm_feat") and not force:
        print("cm_feat already exists (set REBUILD_CM = True to refresh)")
        return
    t0 = time.time()
    loyalty_sql = (f"CASE WHEN nps_valid <= 6 THEN {NPS_LOYALTY_POINTS['detractor']} "
                   f"WHEN nps_valid <= 8 THEN {NPS_LOYALTY_POINTS['passive']} "
                   f"WHEN nps_valid IS NOT NULL THEN {NPS_LOYALTY_POINTS['promoter']} END")
    # ---- interaction-level feedback fact ----
    con.execute(f"""
    CREATE OR REPLACE TABLE fb_i AS
    WITH v AS (
        SELECT interaction_id, customer_id, mth, ts, channel, interaction_type, resolution_status, fcr, sentiment_label,
               csat AS csat_raw, nps AS nps_raw,
               CASE WHEN csat BETWEEN {CSAT_MIN} AND {CSAT_MAX} AND csat = round(csat) THEN csat END csat_valid,
               CASE WHEN nps BETWEEN {NPS_MIN} AND {NPS_MAX} AND nps = round(nps) THEN nps END nps_valid
        FROM itx WHERE in_window)
    SELECT *,
           (csat_raw IS NOT NULL AND csat_valid IS NULL) csat_invalid_flag,
           (nps_raw IS NOT NULL AND nps_valid IS NULL)   nps_invalid_flag,
           (csat_valid - {CSAT_MIN}) / {CSAT_MAX - CSAT_MIN}.0 * 100 csat_normalized,        -- (CSAT-1)/4*100, NULL stays NULL
           CASE WHEN nps_valid <= 6 THEN 'detractor' WHEN nps_valid <= 8 THEN 'passive'
                WHEN nps_valid IS NOT NULL THEN 'promoter' END nps_category,
           {loyalty_sql} nps_loyalty_points
    FROM v""")
    con.execute(f"COPY fb_i TO '{(SILVER_DIR / 'fact_feedback_interaction.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")

    # ---- customer-month feedback aggregation ----
    con.execute(f"""
    CREATE OR REPLACE TABLE fb_m AS
    SELECT customer_id, mth,
        count(*) int_count,
        count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)}) complaint_count,
        count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)}) unresolved_count,
        count_if(fcr) fcr_count,
        count_if(sentiment_label = 'positive') sent_pos_count,
        count_if(sentiment_label = 'neutral')  sent_neu_count,
        count_if(sentiment_label = 'negative') sent_neg_count,
        count_if(csat_invalid_flag) csat_invalid_count,
        count(csat_valid) csat_response_count,
        avg(csat_valid) avg_csat_rating,
        arg_max(csat_valid, ts) FILTER (WHERE csat_valid IS NOT NULL) latest_csat_rating,
        avg(csat_normalized) csat_health_score,
        count_if(csat_valid <= 2) csat_low_count,
        count_if(csat_valid >= 4) csat_high_count,
        count_if(nps_invalid_flag) nps_invalid_count,
        count(nps_valid) nps_response_count,
        avg(nps_valid) avg_nps_rating,
        arg_max(nps_valid, ts) FILTER (WHERE nps_valid IS NOT NULL) latest_nps_rating,
        count_if(nps_category = 'detractor') nps_detractor_count,
        count_if(nps_category = 'passive')   nps_passive_count,
        count_if(nps_category = 'promoter')  nps_promoter_count,
        avg(nps_loyalty_points) nps_loyalty_score
    FROM fb_i GROUP BY ALL""")

    con.execute(f"""
    CREATE OR REPLACE TABLE eng_f AS
    SELECT customer_id, mth,
        count_if(event_type IN {sql_list(ENG_POSITIVE)}) eng_positive_count,
        count_if(event_type IN {sql_list(ENG_NEUTRAL)})  eng_neutral_count,
        count_if(event_type IN {sql_list(ENG_NEGATIVE)}) eng_negative_count,
        count_if(event_type IN {sql_list(UNSUBSCRIBE_EVENTS)}) unsubscribe_count,
        count_if(event_type = 'push_dismiss') push_dismiss_count,
        avg(dur) FILTER (WHERE dur > 0) avg_session_sec
    FROM eng WHERE in_window GROUP BY ALL""")

    con.execute(f"""
    CREATE OR REPLACE TABLE tx_f AS
    SELECT customer_id, mth,
        count_if(transaction_type IN {sql_list(PAYMENT_TYPES)}) payment_attempt_count,
        count_if(transaction_type IN {sql_list(PAYMENT_TYPES)} AND transaction_status = '{TX_SUCCESS}') payment_success_count,
        count_if(transaction_type IN {sql_list(PAYMENT_TYPES)} AND transaction_status = '{TX_FAILED}')  payment_failed_count,
        count_if(transaction_type IN {sql_list(REFUND_TYPES)} AND transaction_status = '{TX_SUCCESS}')  refund_completed_count,
        count(DISTINCT currency) n_currencies
    FROM tx WHERE in_window GROUP BY ALL""")

    con.execute(f"""
    CREATE OR REPLACE TABLE cm_feat AS
    WITH tr AS (   -- trend baselines over the full month spine (prior 3 months, observed months only)
        SELECT customer_id, mth,
               avg(CASE WHEN has_tx THEN gross_spend END) OVER w3 spend_base_3m,
               count(CASE WHEN has_tx THEN 1 END)         OVER w3 spend_base_months,
               avg(CASE WHEN has_eng THEN n_eng END)      OVER w3 eng_base_3m,
               count(CASE WHEN has_eng THEN 1 END)        OVER w3 eng_base_months
        FROM cm
        WINDOW w3 AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING)),
    j AS (
        SELECT m.customer_id, m.mth, m.lifecycle_status, m.months_observed, m.active_months_to_date, m.n_sources, m.source_combo,
               -- customers (context only - never lowers a score)
               c.customer_segment, c.loyalty_tier,
               CASE WHEN c.signup IS NOT NULL THEN date_diff('month', c.signup, last_day(m.mth)) END tenure_months_asof,
               -- source presence (0 = no record, NOT zero activity)
               CAST(m.has_int AS INT) has_interactions, CAST(m.has_eng AS INT) has_engagements, CAST(m.has_tx AS INT) has_transactions,
               -- interactions
               f.int_count, f.complaint_count,
               f.complaint_count * 1.0 / f.int_count complaint_rate,
               f.unresolved_count, f.unresolved_count * 1.0 / f.int_count unresolved_rate,
               f.fcr_count * 1.0 / f.int_count fcr_rate,
               f.sent_pos_count * 1.0 / f.int_count sentiment_pos_share,     -- label-derived: EDA / evaluation only
               f.sent_neu_count * 1.0 / f.int_count sentiment_neu_share,
               f.sent_neg_count * 1.0 / f.int_count sentiment_neg_share,
               -- CSAT
               CAST(coalesce(f.csat_response_count, 0) > 0 AS INT) csat_available,
               coalesce(f.csat_response_count, 0) csat_response_count, coalesce(f.csat_invalid_count, 0) csat_invalid_count,
               f.avg_csat_rating, f.latest_csat_rating, f.csat_health_score,
               CASE WHEN f.csat_response_count > 0 THEN CAST(f.csat_low_count > 0 AS INT) END csat_low_flag,    -- any rating <= 2
               CASE WHEN f.csat_response_count > 0 THEN CAST(f.csat_high_count > 0 AS INT) END csat_high_flag,  -- any rating >= 4
               -- NPS
               CAST(coalesce(f.nps_response_count, 0) > 0 AS INT) nps_available,
               coalesce(f.nps_response_count, 0) nps_response_count, coalesce(f.nps_invalid_count, 0) nps_invalid_count,
               f.avg_nps_rating, f.latest_nps_rating,
               coalesce(f.nps_detractor_count, 0) nps_detractor_count, coalesce(f.nps_passive_count, 0) nps_passive_count,
               coalesce(f.nps_promoter_count, 0) nps_promoter_count,
               f.nps_detractor_count * 1.0 / nullif(f.nps_response_count, 0) nps_detractor_share,
               f.nps_passive_count   * 1.0 / nullif(f.nps_response_count, 0) nps_passive_share,
               f.nps_promoter_count  * 1.0 / nullif(f.nps_response_count, 0) nps_promoter_share,
               f.nps_loyalty_score,
               (f.nps_promoter_count - f.nps_detractor_count) * 100.0 / nullif(f.nps_response_count, 0) nps_net_score,
               -- engagements
               m.n_eng, m.n_logins login_count, m.n_conversions, m.n_conversions * 1.0 / m.n_eng conversion_rate,
               m.eng_seconds, e.avg_session_sec, m.pages_viewed,
               e.eng_positive_count, e.eng_neutral_count, e.eng_negative_count, e.unsubscribe_count, e.push_dismiss_count,
               e.eng_positive_count * 1.0 / nullif(e.eng_positive_count + e.eng_negative_count, 0) positive_engagement_share,

               -- transactions
               m.n_tx, m.n_tx_completed, m.n_tx_failed, m.n_tx_reversed reversal_count, m.n_refunds refund_count,
               m.n_tx_completed * 1.0 / m.n_tx tx_success_rate, m.n_tx_failed * 1.0 / m.n_tx tx_failed_rate,
               t.payment_attempt_count, t.payment_success_count, t.payment_failed_count, t.refund_completed_count,
               t.payment_failed_count * 1.0 / nullif(t.payment_attempt_count, 0) failed_payment_rate,   

               -- Fee related features
               m.n_fees, m.n_fees_completed, m.n_fees_failed, m.n_fees_reversed, m.fee_completed_amt, m.fee_reversed_amt, m.net_fee_amt,

               -- Currency and spend features
               m.tx_currency, t.n_currencies, m.gross_spend gross_spend_native, m.refund_amt refund_amt_native,
               m.gross_spend - m.refund_amt net_spend_native, m.total_customer_outflow, m.spend_pctile,                
                
               -- trends vs own history
               tr.spend_base_3m, tr.spend_base_months,
               CASE WHEN tr.spend_base_months >= 2 AND tr.spend_base_3m > 0 AND (m.has_tx OR {str(TREAT_MISSING_TX_AS_ZERO).upper()})
                    THEN (coalesce(m.gross_spend, 0) - tr.spend_base_3m) / tr.spend_base_3m END spend_change_vs_base,
               tr.eng_base_3m, tr.eng_base_months,
               CASE WHEN tr.eng_base_months >= 2 AND tr.eng_base_3m > 0 AND m.has_eng
                    THEN (m.n_eng - tr.eng_base_3m) / tr.eng_base_3m END eng_change_vs_base
        FROM cm m
        JOIN cust1 c USING (customer_id)
        JOIN tr USING (customer_id, mth)
        LEFT JOIN fb_m f USING (customer_id, mth)
        LEFT JOIN eng_f e USING (customer_id, mth)
        LEFT JOIN tx_f t USING (customer_id, mth)
        WHERE m.active)
    SELECT *, CASE WHEN has_engagements = 1 THEN percent_rank() OVER (PARTITION BY mth, has_engagements ORDER BY n_eng) END eng_pctile
    FROM j""")
    con.execute(f"COPY cm_feat TO '{(SILVER_DIR / 'customer_month_features.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
    print(f"cm_feat: {q('SELECT count(*) FROM cm_feat').iloc[0, 0]:,} active customer-months in {time.time() - t0:,.0f}s")


build_cm_features(force=REBUILD_CM)

FEATURE_GROUPS = {
    "customers (context only)": ["customer_segment", "loyalty_tier", "tenure_months_asof"],
    "source presence": ["has_interactions", "has_engagements", "has_transactions", "n_sources", "source_combo"],
    "interactions": ["int_count", "complaint_count", "complaint_rate", "unresolved_count", "unresolved_rate", "fcr_rate"],
    "interactions (label-derived: EDA/evaluation only)": ["sentiment_pos_share", "sentiment_neu_share", "sentiment_neg_share"],
    "CSAT": ["csat_available", "csat_response_count", "csat_invalid_count", "avg_csat_rating", "latest_csat_rating",
             "csat_health_score", "csat_low_flag", "csat_high_flag"],
    "NPS": ["nps_available", "nps_response_count", "nps_invalid_count", "avg_nps_rating", "latest_nps_rating",
            "nps_detractor_count", "nps_passive_count", "nps_promoter_count", "nps_detractor_share", "nps_passive_share",
            "nps_promoter_share", "nps_loyalty_score", "nps_net_score"],
    "engagements": ["n_eng", "login_count", "conversion_rate", "eng_seconds", "avg_session_sec", "pages_viewed",
                    "eng_positive_count", "eng_neutral_count", "eng_negative_count", "unsubscribe_count",
                    "positive_engagement_share", "eng_pctile"],
    "transactions": ["n_tx", "tx_success_rate", "tx_failed_rate", "failed_payment_rate", "payment_attempt_count",
                     "refund_count", "reversal_count", "n_fees", "n_fees_completed", "n_fees_failed", "n_fees_reversed", "fee_completed_amt", 
                     "fee_reversed_amt", "net_fee_amt", "tx_currency", "n_currencies", "gross_spend_native",
                     "refund_amt_native", "net_spend_native", "total_customer_outflow", "spend_pctile"],
    "trends (own history)": ["spend_base_3m", "spend_change_vs_base", "eng_base_3m", "eng_change_vs_base"],
}
rows = []
n_all = q("SELECT count(*) FROM cm_feat").iloc[0, 0]
for g, cols in FEATURE_GROUPS.items():
    for c in cols:
        r = q(f"SELECT count({c}) nn, avg(try_cast({c} AS DOUBLE)) mean FROM cm_feat").iloc[0]
        rows.append({"group": g, "feature": c, "pct_non_null": round(100 * r.nn / n_all, 2),
                     "mean": None if pd.isna(r["mean"]) else round(float(r["mean"]), 4)})
save(pd.DataFrame(rows), "s4a_feature_dictionary_and_coverage", show=70)
flag("S4A", f"cm_feat built with {sum(len(v) for v in FEATURE_GROUPS.values())} documented features for {n_all:,} active "
            "customer-months; survey features are NULL when not observed (no imputation)")

## Step 1 - Structure & grain

In [ ]:
def step1_structure():
    header("STEP 1 - Structure, grain, duplicates, join keys")
    rows = []
    for t, pk in TABLES.items():
        schema = q(f"DESCRIBE raw_{t}")[["column_name", "column_type"]]
        save(schema.assign(table=t), f"s1_schema_{t}", show=0)
        r = q(f"""SELECT count(*) n_rows, count(DISTINCT {pk}) n_distinct_pk,
                         count(DISTINCT customer_id) n_customers FROM raw_{t}""").iloc[0]
        full_dup = q(f"SELECT (SELECT count(*) FROM raw_{t}) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_{t}))").iloc[0, 0]
        rows.append({"table": t, "n_rows": r.n_rows, "n_cols": len(schema), "primary_key": pk,
                     "pk_duplicates": r.n_rows - r.n_distinct_pk, "full_row_duplicates": full_dup,
                     "n_customers": r.n_customers,
                     "rows_per_customer": round(r.n_rows / max(r.n_customers, 1), 2),
                     "grain": "1 row = 1 customer" if t == "customers" else f"1 row = 1 {t[:-1]} event"})
        if r.n_rows - r.n_distinct_pk > 0:
            flag("S1", f"{t}: {r.n_rows - r.n_distinct_pk:,} duplicate {pk} values")
        if full_dup > 0:
            flag("S1", f"{t}: {full_dup:,} fully duplicated rows")
    save(pd.DataFrame(rows), "s1_table_overview")

    # Date range, unparseable timestamps, out-of-window rows, month column consistency
    rows = []
    for t, v in EVENT_VIEWS.items():
        r = q(f"""
            SELECT min(ts) min_ts, max(ts) max_ts,
                   count_if(ts IS NULL) unparseable_ts,
                   count_if(NOT in_window) outside_window,
                   count_if(left(CAST(month AS VARCHAR), 7) <> strftime(ts, '%Y-%m')) month_col_mismatch,
                   count(DISTINCT mth) n_months
            FROM {v}""").iloc[0].to_dict()
        rows.append({"table": t, **r})
        if r["outside_window"] > 0:
            flag("S1", f"{t}: {r['outside_window']:,} rows outside {WINDOW_START}..{WINDOW_END}")
        if r["month_col_mismatch"] > 0:
            flag("S1", f"{t}: {r['month_col_mismatch']:,} rows where 'month' != month of {TS_COL[t]}")
    save(pd.DataFrame(rows), "s1_date_ranges")

    # Referential integrity (orphan customer_ids)
    rows = []
    for t in EVENT_VIEWS:
        r = q(f"""
            SELECT count(DISTINCT e.customer_id) n_customers,
                   count(DISTINCT e.customer_id) FILTER (WHERE c.customer_id IS NULL) orphan_customers,
                   count_if(c.customer_id IS NULL) orphan_rows,
                   count_if(e.customer_id IS NULL) null_customer_id_rows
            FROM raw_{t} e LEFT JOIN (SELECT DISTINCT customer_id FROM raw_customers) c
              ON e.customer_id = c.customer_id""").iloc[0].to_dict()
        rows.append({"table": t, **r, "pct_customers_found": round(100 * (1 - r["orphan_customers"] / max(r["n_customers"], 1)), 2)})
        if r["orphan_customers"] > 0:
            flag("S1", f"{t}: {r['orphan_customers']:,} customer_ids not in customers.csv")
    save(pd.DataFrame(rows), "s1_orphan_ids")

    # Customers with no activity in any source
    r = q("""
        SELECT count(*) n_customers,
               count_if(customer_id NOT IN (SELECT customer_id FROM raw_transactions)
                    AND customer_id NOT IN (SELECT customer_id FROM raw_engagements)
                    AND customer_id NOT IN (SELECT customer_id FROM raw_interactions)) no_activity
        FROM (SELECT DISTINCT customer_id FROM raw_customers)""").iloc[0]
    pct = 100 * r.no_activity / r.n_customers
    save(pd.DataFrame([{"n_customers": r.n_customers, "customers_without_any_activity": r.no_activity,
                        "pct": round(pct, 2)}]), "s1_customers_without_activity")
    if pct > 10:
        flag("S1", f"{pct:.1f}% of customers have no activity in any source in the period")


run_step(step1_structure)

## Step 2 - Data quality

In [ ]:
def step2_quality():
    header("STEP 2 - Nulls, invalid values, statuses, outliers")

    # 2a. Null / blank % for every column of every table
    frames = []
    for t in TABLES:
        cs = columns_of(f"raw_{t}")
        expr = ", ".join(
            f"""round(100.0 * count_if("{c}" IS NULL OR trim(CAST("{c}" AS VARCHAR)) IN
                ('', 'NA', 'N/A', 'null', 'NULL', 'None', 'nan')) / count(*), 3) AS "{c}" """ for c in cs)
        d = q(f"SELECT {expr} FROM raw_{t}").T.reset_index()
        d.columns = ["column", "null_or_blank_pct"]
        frames.append(d.assign(table=t))
    nulls = save(pd.concat(frames)[["table", "column", "null_or_blank_pct"]]
                 .sort_values("null_or_blank_pct", ascending=False), "s2_null_profile", show=20)
    for _, r in nulls[nulls.null_or_blank_pct > 5].iterrows():
        flag("S2", f"{r.table}.{r.column}: {r.null_or_blank_pct:.1f}% null/blank - check if expected")

    # 2b. Do nulls cluster in certain months? (key sparse columns)
    save(q("""
        SELECT mth, count(*) n,
               round(100.0*count_if(csat IS NULL)/count(*), 2) csat_null_pct,
               round(100.0*count_if(nps IS NULL)/count(*), 2)  nps_null_pct,
               round(100.0*count_if(txt IS NULL OR trim(txt) = '')/count(*), 2) text_blank_pct,
               round(100.0*count_if(sscore IS NULL)/count(*), 2) sentiment_gt_null_pct
        FROM itx GROUP BY 1 ORDER BY 1"""), "s2_nulls_by_month_interactions", show=12)
    save(q("""
        SELECT mth, count(*) n,
               round(100.0*count_if(campaign_id IS NULL OR trim(CAST(campaign_id AS VARCHAR))='')/count(*), 2) campaign_null_pct,
               round(100.0*count_if(app_version IS NULL OR trim(CAST(app_version AS VARCHAR))='')/count(*), 2) app_version_null_pct
        FROM eng GROUP BY 1 ORDER BY 1"""), "s2_nulls_by_month_engagements", show=0)

    # 2c. Invalid-value rules
    rules = [
        ("transactions", "tx", "amount is null", "amt IS NULL"),
        ("transactions", "tx", "amount < 0", "amt < 0"),
        ("transactions", "tx", "amount = 0", "amt = 0"),
        ("transactions", "tx", "discount < 0", "disc < 0"),
        ("transactions", "tx", "discount > amount", "disc > abs(amt)"),
        ("transactions", "tx", "future timestamp", "ts > now()"),
        ("engagements", "eng", "session_duration < 0", "dur < 0"),
        ("engagements", "eng", "session_duration > 4h", "dur > 14400"),
        ("engagements", "eng", "pages_viewed < 0", "pages < 0"),
        ("interactions", "itx", "duration < 0", "dur < 0"),
        ("interactions", "itx", "duration > 4h", "dur > 14400"),
        ("interactions", "itx", "csat outside 1-5 (assumed scale)", "csat NOT BETWEEN 1 AND 5"),
        ("interactions", "itx", "nps outside 0-10", "nps NOT BETWEEN 0 AND 10"),
        ("interactions", "itx", "sentiment_score_gt outside [-1,1]", "sscore NOT BETWEEN -1 AND 1"),
        ("interactions", "itx", "label positive but score < 0", "sentiment_label='positive' AND sscore < 0"),
        ("interactions", "itx", "label negative but score > 0", "sentiment_label='negative' AND sscore > 0"),
        ("interactions", "itx", "text blank", "txt IS NULL OR trim(txt) = ''"),
        ("customers", "cust", "age < 18 or > 100", "age_n < 18 OR age_n > 100"),
        ("customers", "cust", "tenure_months < 0", "tenure_n < 0"),
        ("customers", "cust", "signup after data window", f"signup >= DATE '{WINDOW_END}'"),
        ("customers", "cust", "signup_date unparseable/null", "signup IS NULL"),
    ]
    rows = []
    for t, v, name, cond in rules:
        n_bad, n = (int(x) for x in q(f"SELECT count_if({cond}) n_bad, count(*) n FROM {v}").iloc[0])
        rows.append({"table": t, "rule": name, "n_violations": n_bad, "pct": round(100 * n_bad / n, 3)})
        if n_bad > 0:
            flag("S2", f"{t}: '{name}' -> {n_bad:,} rows ({100 * n_bad / n:.2f}%)")
    save(pd.DataFrame(rows), "s2_invalid_value_rules", show=30)

    # 2d. Transaction status x type (defines gross vs net spend)
    save(q("""
        SELECT transaction_type, transaction_status, count(*) n,
               round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_rows,
               count_if(amt < 0) n_negative_amount, round(median(amt), 2) median_amount
        FROM tx GROUP BY ALL ORDER BY n DESC"""), "s2_tx_status_by_type", show=20)
    save(q("""
        SELECT currency, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct,
               count(DISTINCT customer_id) n_customers, round(median(amt), 2) median_amount
        FROM tx GROUP BY 1 ORDER BY n DESC"""), "s2_tx_currency_mix")
    n_cur = q("SELECT count(DISTINCT currency) FROM tx").iloc[0, 0]
    if n_cur > 1:
        flag("S2", f"{n_cur} currencies in transactions - do NOT sum amounts across currencies; "
                   "use FX conversion or within-currency percentiles")
    multi = q("""SELECT count(*) FROM (SELECT customer_id FROM tx GROUP BY 1
                 HAVING count(DISTINCT currency) > 1)""").iloc[0, 0]
    if multi > 0:
        flag("S2", f"{multi:,} customers transact in more than one currency")
    save(q("""SELECT resolution_status, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s2_resolution_status")

    # 2e. Outliers - per-customer annual volumes (percentiles, not mean)
    rows = []
    for t, v in EVENT_VIEWS.items():
        r = q(f"""
            WITH c AS (SELECT customer_id, count(*) n FROM {v} WHERE in_window GROUP BY 1)
            SELECT '{t}' source, count(*) customers_with_activity, min(n) min,
                   quantile_cont(n, 0.5) p50, quantile_cont(n, 0.95) p95,
                   quantile_cont(n, 0.99) p99, max(n) max FROM c""")
        rows.append(r)
    save(pd.concat(rows), "s2_events_per_customer_percentiles")
    save(q("""WITH p AS (SELECT currency, quantile_cont(amt, 0.99) p99 FROM tx GROUP BY 1)
              SELECT t.currency, round(quantile_cont(t.amt, 0.01), 2) p01, round(median(t.amt), 2) p50,
                     round(quantile_cont(t.amt, 0.95), 2) p95, round(any_value(p.p99), 2) p99,
                     round(max(t.amt), 2) max, count_if(t.amt > 5 * p.p99) n_above_5x_p99
              FROM tx t JOIN p USING (currency) GROUP BY 1 ORDER BY 1"""), "s2_amount_outliers_by_currency")
    save(q("""SELECT customer_id, count(*) n_interactions, count_if(interaction_type = 'complaint') n_complaints
              FROM itx WHERE in_window GROUP BY 1 ORDER BY n_interactions DESC LIMIT 50"""),
         "s2_top_customers_by_interactions", show=5)


run_step(step2_quality)

## Step 3 - Univariate profile

In [ ]:
NUMERIC = {"tx": ["amt", "disc"], "eng": ["dur", "pages"],
           "itx": ["dur", "csat", "nps", "sscore"], "cust": ["age_n", "tenure_n"]}
CATEGORICAL = {
    "tx": ["transaction_type", "product_category", "currency", "payment_method", "merchant_channel",
           "transaction_status", "is_international"],
    "eng": ["engagement_channel", "event_type", "device_type", "app_version", "is_converted"],
    "itx": ["channel", "interaction_type", "resolution_status", "first_contact_resolution", "sentiment_label"],
    "cust": ["gender", "region", "country", "customer_segment", "acquisition_channel", "income_band",
             "preferred_channel", "loyalty_tier", "marketing_consent"],
}


def step3_univariate():
    header("STEP 3 - Univariate distributions")
    frames = []
    for v, cs in NUMERIC.items():
        for c in cs:
            frames.append(q(f"""
                SELECT '{v}' tbl, '{c}' col, count({c}) n_nonnull,
                       round(100.0*count_if({c} = 0)/count(*), 2) zero_pct,
                       round(100.0*count_if({c} < 0)/count(*), 2) neg_pct,
                       min({c}) min, quantile_cont({c}, 0.01) p01, quantile_cont({c}, 0.25) p25,
                       median({c}) p50, quantile_cont({c}, 0.75) p75, quantile_cont({c}, 0.95) p95,
                       quantile_cont({c}, 0.99) p99, max({c}) max, avg({c}) mean, stddev({c}) std,
                       skewness({c}) skew FROM {v}"""))
    num = save(pd.concat(frames), "s3_numeric_profile", show=20)
    for _, r in num.iterrows():
        if pd.notna(r["skew"]) and abs(r["skew"]) > 1:
            flag("S3", f"{r.tbl}.{r.col} skew={r['skew']:.2f} -> consider log1p / capping at P99")

    frames, summary = [], []
    for v, cs in CATEGORICAL.items():
        for c in cs:
            d = q(f"""SELECT CAST({c} AS VARCHAR) value, count(*) n,
                             round(100.0*count(*)/sum(count(*)) OVER (), 3) pct
                      FROM {v} GROUP BY 1 ORDER BY n DESC""")
            d.insert(0, "col", c)
            d.insert(0, "tbl", v)
            frames.append(d)
            summary.append({"tbl": v, "col": c, "n_levels": len(d),
                            "n_rare_levels": int((d.pct < RARE_CAT_PCT).sum()),
                            "top_level": d.value.iloc[0], "top_pct": d.pct.iloc[0]})
    save(pd.concat(frames), "s3_categorical_frequencies", show=0)
    cat = save(pd.DataFrame(summary), "s3_categorical_summary", show=30)
    for _, r in cat[cat.n_rare_levels > 0].iterrows():
        flag("S3", f"{r.tbl}.{r.col}: {r.n_rare_levels} rare levels (<{RARE_CAT_PCT}%) -> group as 'Other'")


run_step(step3_univariate)

## Step 4 - Time patterns

In [ ]:
def step4_time():
    header("STEP 4 - Monthly trends, seasonality, gaps, partial months")
    monthly = q(f"""
        WITH t AS (SELECT mth, count(*) tx_rows, count(DISTINCT customer_id) tx_customers,
                          round(100.0*count_if(transaction_status <> 'completed')/count(*), 2) tx_failed_or_reversed_pct,
                          round(sum(amt) FILTER (WHERE transaction_status='completed'
                                AND transaction_type IN {sql_list(SPEND_TYPES)} AND currency='USD'), 0) usd_gross_spend
                   FROM tx WHERE in_window GROUP BY 1),
             e AS (SELECT mth, count(*) eng_rows, count(DISTINCT customer_id) eng_customers
                   FROM eng WHERE in_window GROUP BY 1),
             i AS (SELECT mth, count(*) int_rows, count(DISTINCT customer_id) int_customers,
                          round(100.0*count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)})/count(*), 2) complaint_pct,
                          round(100.0*count_if(sentiment_label='negative')/count(*), 2) negative_pct,
                          round(avg(csat), 3) avg_csat, round(avg(nps), 3) avg_nps
                   FROM itx WHERE in_window GROUP BY 1)
        SELECT * FROM t FULL JOIN e USING (mth) FULL JOIN i USING (mth) ORDER BY mth""")
    save(monthly, "s4_monthly_trends", show=12)

    vol_cols = ["tx_rows", "tx_customers", "usd_gross_spend", "eng_rows", "eng_customers", "int_rows", "int_customers"]
    mom = monthly[["mth"]].copy()
    for c in vol_cols:
        mom[c + "_mom_pct"] = (monthly[c].pct_change() * 100).round(2)
    save(mom, "s4_mom_change_pct", show=12)
    for c in vol_cols:
        big = mom[mom[c + "_mom_pct"].abs() > MOM_ALERT_PCT]
        for _, r in big.iterrows():
            flag("S4", f"{c}: {r[c + '_mom_pct']:+.1f}% MoM in {pd.Timestamp(r.mth):%Y-%m} -> real event or load issue?")

    # Days with data per month (partial months / gaps)
    frames = []
    for t, v in EVENT_VIEWS.items():
        frames.append(q(f"""
            SELECT '{t}' source, mth, count(DISTINCT CAST(ts AS DATE)) days_with_data,
                   day(last_day(mth)) calendar_days, min(ts) first_ts, max(ts) last_ts
            FROM {v} WHERE in_window GROUP BY 1, 2 ORDER BY 2"""))
    days = save(pd.concat(frames), "s4_days_with_data", show=0)
    for _, r in days[days.days_with_data < days.calendar_days].iterrows():
        flag("S4", f"{r.source} {pd.Timestamp(r.mth):%Y-%m}: data on {r.days_with_data}/{r.calendar_days} days (gap/partial month)")

    # Daily anomalies: day volume < 50% or > 200% of trailing 7-day median
    frames = []
    for t, v in EVENT_VIEWS.items():
        d = q(f"SELECT CAST(ts AS DATE) d, count(*) n FROM {v} WHERE in_window GROUP BY 1 ORDER BY 1")
        d["rolling_median_7d"] = d.n.rolling(7, min_periods=3).median().shift(1)
        d["ratio"] = (d.n / d.rolling_median_7d).round(2)
        frames.append(d[(d.ratio < 0.5) | (d.ratio > 2.0)].assign(source=t))
    anomalies = save(pd.concat(frames), "s4_daily_volume_anomalies", show=10)
    if len(anomalies):
        flag("S4", f"{len(anomalies)} days with abnormal volume (see s4_daily_volume_anomalies.csv)")

    # Day-of-week / hour patterns (seasonality within week)
    frames = []
    for t, v in EVENT_VIEWS.items():
        frames.append(q(f"""SELECT '{t}' source, dayname(ts) day_of_week, isodow(ts) dow, count(*) n
                            FROM {v} WHERE in_window GROUP BY ALL ORDER BY dow"""))
    save(pd.concat(frames), "s4_day_of_week", show=0)

    # Chart
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    x = pd.to_datetime(monthly.mth).dt.strftime("%Y-%m")
    for c in ["tx_rows", "eng_rows", "int_rows"]:
        ax[0].plot(x, monthly[c], marker="o", label=c)
    for c in ["tx_customers", "eng_customers", "int_customers"]:
        ax[1].plot(x, monthly[c], marker="o", label=c)
    ax[0].set_title("Monthly rows by source"); ax[1].set_title("Monthly active customers by source")
    for a in ax:
        a.tick_params(axis="x", rotation=45); a.legend(); a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "s4_monthly_trends.png", dpi=120); finish_plot()


run_step(step4_time)

## Step 5 - Source coverage

In [ ]:
def step5_coverage():
    header("STEP 5 - Coverage matrix across Transactions / Engagements / Interactions")
    save(q("""SELECT n_sources, count(*) n_customer_months,
                     round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_of_all_customer_months
              FROM cm WHERE NOT pre_signup GROUP BY 1 ORDER BY 1"""), "s5_n_sources_all_customer_months")
    combo = save(q("""SELECT source_combo, count(*) n_customer_months,
                             round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_of_active
                      FROM cm WHERE active GROUP BY 1 ORDER BY n_customer_months DESC"""), "s5_source_combinations")
    by_month = save(q("""
        SELECT mth, count(*) all_customers, count_if(active) active_customers,
               round(100.0*count_if(active)/count(*), 2) pct_active,
               round(100.0*count_if(n_sources = 3)/nullif(count_if(active), 0), 2) pct_active_3_sources,
               round(100.0*count_if(n_sources = 2)/nullif(count_if(active), 0), 2) pct_active_2_sources,
               round(100.0*count_if(n_sources = 1)/nullif(count_if(active), 0), 2) pct_active_1_source
        FROM cm WHERE NOT pre_signup GROUP BY 1 ORDER BY 1"""), "s5_coverage_by_month", show=12)
    save(q("""SELECT months_with_2plus_sources, count(*) n_customers,
                     round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
              FROM (SELECT customer_id, count_if(n_sources >= 2) months_with_2plus_sources FROM cm GROUP BY 1)
              GROUP BY 1 ORDER BY 1"""), "s5_customers_by_months_with_2plus_sources", show=13)

    pct2 = q("SELECT 100.0*count_if(n_sources >= 2)/count(*) FROM cm WHERE active").iloc[0, 0]
    flag("S5", f"{pct2:.1f}% of active customer-months have >= 2 sources "
               f"({'good' if pct2 >= 50 else 'sparse -> confidence must reflect missing sources'})")
    avg_active = by_month.pct_active.mean()
    flag("S5", f"On average {avg_active:.1f}% of customers are active in a given month")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(combo.source_combo, combo.pct_of_active)
    ax.set_xlabel("% of active customer-months"); ax.set_title("Source combinations (active customer-months)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s5_source_combinations.png", dpi=120); finish_plot()


run_step(step5_coverage)

## Step 6 - Lifecycle

In [ ]:
def step6_lifecycle():
    header("STEP 6 - Lifecycle: first/last seen, active months, status, MoM change")
    per_cust = save(q("""
        WITH c AS (SELECT customer_id, sum(CAST(active AS INT)) n_active_months FROM cm GROUP BY 1)
        SELECT n_active_months, count(*) n_customers, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
        FROM c GROUP BY 1 ORDER BY 1"""), "s6_active_months_distribution", show=13)
    save(q("""SELECT first_active_mth, count(DISTINCT customer_id) n_customers FROM cm
              WHERE first_active_mth IS NOT NULL GROUP BY 1 ORDER BY 1"""), "s6_first_active_month", show=0)
    save(q("""WITH c AS (SELECT customer_id, max(CASE WHEN active THEN mth END) last_active FROM cm GROUP BY 1)
              SELECT last_active, count(*) n_customers FROM c WHERE last_active IS NOT NULL GROUP BY 1 ORDER BY 1"""),
         "s6_last_active_month", show=0)
    flag("S6", f"Customers first seen in {WINDOW_START[:7]} are left-censored (history before the window is unknown)")

    status = q("""SELECT mth, lifecycle_status, count(*) n FROM cm WHERE NOT pre_signup GROUP BY ALL""")
    save(status.pivot(index="mth", columns="lifecycle_status", values="n").fillna(0).astype(int).reset_index(),
         "s6_lifecycle_status_by_month", show=12)

    lt3 = per_cust[(per_cust.n_active_months > 0) & (per_cust.n_active_months < MIN_HISTORY_MONTHS)].n_customers.sum()
    act = per_cust[per_cust.n_active_months > 0].n_customers.sum()
    flag("S6", f"{100 * lt3 / max(act, 1):.1f}% of active customers have < {MIN_HISTORY_MONTHS} active months "
               "-> weak trend history")

    # MoM change buckets (customer active in consecutive months)
    save(q("""
        WITH x AS (SELECT customer_id, mth, gross_spend, n_eng, n_int,
                          lag(gross_spend) OVER w p_spend, lag(n_eng) OVER w p_eng
                   FROM cm WINDOW w AS (PARTITION BY customer_id ORDER BY mth)),
        y AS (SELECT 'gross_spend' metric, (coalesce(gross_spend,0) - p_spend) / p_spend chg FROM x WHERE p_spend > 0
              UNION ALL
              SELECT 'engagement_events', (coalesce(n_eng,0) - p_eng) / p_eng FROM x WHERE p_eng > 0)
        SELECT metric, count(*) n_pairs,
               round(100.0*count_if(chg <= -1)/count(*), 2) pct_dropped_to_zero,
               round(100.0*count_if(chg > -1 AND chg <= -0.3)/count(*), 2) pct_decline_gt_30,
               round(100.0*count_if(chg > -0.3 AND chg < 0.3)/count(*), 2) pct_stable,
               round(100.0*count_if(chg >= 0.3)/count(*), 2) pct_increase_gt_30,
               round(median(chg), 3) median_change
        FROM y GROUP BY 1"""), "s6_mom_change_buckets")

    # Signup vs activity consistency
    save(q("""
        SELECT count(DISTINCT customer_id) FILTER (WHERE pre_signup AND active) customers_active_before_signup,
               count(*) FILTER (WHERE pre_signup AND active) customer_months_active_before_signup
        FROM cm"""), "s6_activity_before_signup")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(per_cust.n_active_months.astype(str), per_cust.n_customers)
    ax.set_xlabel("Active months in period"); ax.set_ylabel("Customers"); ax.set_title("History depth per customer")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s6_active_months_hist.png", dpi=120); finish_plot()


run_step(step6_lifecycle)

## Step 7 - Interactions / sentiment (clause analyses now QA only - not modelling features)

In [ ]:
def step7_interactions():
    header("STEP 7 - Interactions, text, CSAT/NPS, sentiment labels")
    save(q("""SELECT channel, interaction_type, count(*) n FROM itx GROUP BY ALL ORDER BY n DESC"""),
         "s7_channel_by_type", show=0)
    save(q("""SELECT interaction_type, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 2) pct_positive,
                     round(100.0*count_if(sentiment_label='neutral')/count(*), 2)  pct_neutral,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 2) pct_negative,
                     round(avg(sscore), 3) avg_sentiment_gt
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_sentiment_by_type")
    save(q("""SELECT channel, count(*) n,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 2) pct_negative,
                     round(100.0*count(csat)/count(*), 2) csat_fill_pct,
                     round(100.0*count(nps)/count(*), 2) nps_fill_pct,
                     round(avg(sscore), 3) avg_sentiment_gt
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_by_channel")

    # Label vs score consistency
    save(q("""SELECT sentiment_label, count(*) n, round(min(sscore),3) min, round(quantile_cont(sscore,0.05),3) p05,
                     round(median(sscore),3) p50, round(quantile_cont(sscore,0.95),3) p95, round(max(sscore),3) max
              FROM itx GROUP BY 1 ORDER BY 1"""), "s7_label_vs_score")

    # Text quality
    txt = save(q("""
        SELECT count(*) n, round(100.0*count_if(txt IS NULL OR trim(txt)='')/count(*), 2) blank_pct,
               round(avg(length(txt)), 1) avg_chars, quantile_cont(length(txt), 0.5) p50_chars,
               quantile_cont(length(txt), 0.95) p95_chars,
               round(100.0*count_if(len(string_split(trim(txt), ' ')) < 3)/count(*), 2) lt3_words_pct,
               round(100.0*count_if(regexp_matches(txt, '[^\\x00-\\x7F]'))/count(*), 2) non_ascii_pct,
               count(DISTINCT txt) n_distinct_texts,
               round(100.0*count(DISTINCT txt)/count(*), 2) distinct_text_pct
        FROM itx"""), "s7_text_quality")
    if txt.distinct_text_pct.iloc[0] < 20:
        flag("S7", f"Only {txt.distinct_text_pct.iloc[0]:.1f}% of texts are unique -> template-like text; "
                   "check for overfitting and memorisation in the sentiment model")
    top = save(q("""SELECT txt, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 3) pct
                    FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 100"""), "s7_top_100_texts", show=5)
    flag("S7", f"Top 100 texts cover {top.pct.sum():.1f}% of interactions")

    # v2: QA ONLY - NOT a modelling feature. Splitting transcripts on commas does not reliably identify topics,
    #     so the clause analyses below are kept for template/QA inspection only and are never used as features.
    save(q("""SELECT trim(split_part(txt, ',', 1)) first_clause, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 1) pct_pos,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 1) pct_neg
              FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 50"""), "s7_first_clause_label_mix_QA_ONLY", show=10)
    save(q("""SELECT trim(split_part(txt, ',', 2)) second_clause, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 1) pct_pos,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 1) pct_neg
              FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 50"""), "s7_second_clause_label_mix_QA_ONLY", show=0)

    # CSAT / NPS as validation labels
    save(q("""SELECT round(100.0*count(csat)/count(*), 2) csat_fill_pct, round(100.0*count(nps)/count(*), 2) nps_fill_pct,
                     round(100.0*count_if(csat IS NOT NULL OR nps IS NOT NULL)/count(*), 2) any_survey_pct,
                     round(corr(sscore, csat), 3) corr_sentiment_csat, round(corr(sscore, nps), 3) corr_sentiment_nps,
                     round(corr(csat, nps), 3) corr_csat_nps
              FROM itx"""), "s7_survey_fill_and_correlation")
    save(q("""SELECT 'csat' metric, CAST(csat AS INT) score, count(*) n FROM itx WHERE csat IS NOT NULL GROUP BY ALL
              UNION ALL
              SELECT 'nps', CAST(nps AS INT), count(*) FROM itx WHERE nps IS NOT NULL GROUP BY ALL
              ORDER BY 1, 2"""), "s7_survey_distributions", show=0)
    save(q("""SELECT sentiment_label, round(avg(csat), 3) avg_csat, round(avg(nps), 3) avg_nps,
                     count(csat) n_csat, count(nps) n_nps
              FROM itx GROUP BY 1 ORDER BY 1"""), "s7_survey_by_sentiment_label")
    save(q("""SELECT resolution_status, count(*) n, round(avg(sscore), 3) avg_sentiment_gt,
                     round(avg(csat), 3) avg_csat, round(100.0*count_if(fcr)/count(*), 2) fcr_pct
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_resolution_vs_sentiment")

    # Stratified sample for manual reading (add your own label in 'my_label')
    sample = q(f"""SELECT interaction_id, channel, interaction_type, resolution_status, txt AS interaction_text,
                          sentiment_label, sscore AS sentiment_score_gt, csat, nps, '' AS my_label, '' AS notes
                   FROM itx QUALIFY row_number() OVER (PARTITION BY sentiment_label ORDER BY hash(interaction_id || '{SEED}'))
                                    <= {TEXT_SAMPLE_PER_LABEL}""")
    save(sample, "s7_manual_review_sample", show=0)

    lens = q("SELECT length(txt) l FROM itx USING SAMPLE 200000 ROWS")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(lens.l.dropna(), bins=50); ax.set_title("Interaction text length (chars, 200k sample)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7_text_length_hist.png", dpi=120); finish_plot()


run_step(step7_interactions)

## Step 7A - CSAT and NPS coverage and missingness analysis (NEW)
Do not assume missing CSAT/NPS is random - this step tests it.

| Analysis | Business question |
|---|---|
| Availability overlap | Are both surveys available for the same interactions? |
| Missingness by interaction type / channel / resolution / sentiment / segment / tier / month | Is survey availability linked to any of these? |
| Response frequency | Do some customers have multiple survey responses? |
| Response propensity model (csat_response_flag, nps_response_flag) | Can observed variables predict whether a survey exists? (AUC ~ 0.5 => no) |

These checks show whether missingness depends on observed variables; they cannot rule out dependence on the
unobserved rating itself.

In [ ]:
def step7a_survey_missingness():
    header("STEP 7A - CSAT/NPS coverage, missingness and response propensity")
    con.execute("""CREATE OR REPLACE TEMP VIEW fb_c AS
                   SELECT f.*, c.customer_segment, c.loyalty_tier FROM fb_i f LEFT JOIN cust1 c USING (customer_id)""")
    # 1. overlap
    ov = q("""SELECT csat_valid IS NOT NULL csat_available, nps_valid IS NOT NULL nps_available, count(*) n,
                     round(100.0*count(*)/sum(count(*)) OVER (), 2) pct FROM fb_i GROUP BY 1, 2 ORDER BY 1, 2""")
    save(ov, "s7a_csat_nps_overlap")
    p_c, p_n = ov[ov.csat_available].pct.sum() / 100, ov[ov.nps_available].pct.sum() / 100
    both = ov[ov.csat_available & ov.nps_available].pct.sum() / 100
    ratio = both / (p_c * p_n) if p_c * p_n else np.nan
    flag("S7A", f"CSAT on {p_c:.1%}, NPS on {p_n:.1%}, both on {both:.2%} of interactions "
                f"(observed/expected-if-independent = {ratio:.2f}) -> "
                f"{'collected independently' if abs(ratio - 1) < 0.05 else 'collection is linked - check survey design'}")

    # 2. coverage / missingness by dimension
    dims = ["interaction_type", "channel", "resolution_status", "sentiment_label", "customer_segment", "loyalty_tier", "mth"]
    frames, summ = [], []
    for d in dims:
        t = q(f"""SELECT CAST({d} AS VARCHAR) level, count(*) n,
                         count(csat_valid) n_csat, round(100.0*count(csat_valid)/count(*), 2) csat_coverage_pct,
                         count(nps_valid) n_nps, round(100.0*count(nps_valid)/count(*), 2) nps_coverage_pct,
                         round(avg(csat_valid), 3) avg_csat, round(avg(nps_valid), 3) avg_nps
                  FROM fb_c GROUP BY 1 ORDER BY 1""")
        frames.append(t.assign(dimension=d))
        row = {"dimension": d, "levels": len(t)}
        for s, col in (("csat", "n_csat"), ("nps", "n_nps")):
            v, p = cramers_v(pd.DataFrame({"yes": t[col], "no": t.n - t[col]}))
            row.update({f"{s}_coverage_spread_pp": round(t[f"{s}_coverage_pct"].max() - t[f"{s}_coverage_pct"].min(), 2),
                        f"{s}_cramers_v": round(v, 4), f"{s}_p_value": p})
        summ.append(row)
    by_dim = pd.concat(frames)[["dimension", "level", "n", "n_csat", "csat_coverage_pct", "n_nps", "nps_coverage_pct",
                                "avg_csat", "avg_nps"]]
    save(by_dim, "s7a_coverage_by_dimension", show=0)
    for d in dims:
        display(Markdown(f"**CSAT/NPS coverage by {d}**")); display(by_dim[by_dim.dimension == d])
    summ = save(pd.DataFrame(summ), "s7a_missingness_summary", show=10)
    for s in ("csat", "nps"):
        bad = summ[(summ[f"{s}_cramers_v"] >= 0.02) | (summ[f"{s}_coverage_spread_pp"] > 1.5)]
        if len(bad):
            flag("S7A", f"{s.upper()} coverage varies with {', '.join(bad.dimension)} -> missingness NOT random on these; report/weight by group")
        else:
            flag("S7A", f"{s.upper()} coverage flat across all {len(dims)} dimensions (max spread "
                        f"{summ[f'{s}_coverage_spread_pp'].max()} pp, max Cramer's V {summ[f'{s}_cramers_v'].max()})")

    # 3. response frequency per customer
    freq = q("""WITH c AS (SELECT customer_id, count(csat_valid) k_csat, count(nps_valid) k_nps FROM fb_i GROUP BY 1)
                SELECT 'csat' survey, k_csat responses, count(*) customers FROM c WHERE k_csat > 0 GROUP BY 1, 2
                UNION ALL
                SELECT 'nps', k_nps, count(*) FROM c WHERE k_nps > 0 GROUP BY 1, 2 ORDER BY 1, 2""")
    freq["pct_of_responding_customers"] = (100 * freq.customers / freq.groupby("survey").customers.transform("sum")).round(2)
    save(freq, "s7a_response_frequency_per_customer", show=20)
    multi = freq[freq.responses >= 2].groupby("survey").pct_of_responding_customers.sum()
    flag("S7A", f"Customers with 2+ responses in 12 months: CSAT {multi.get('csat', 0):.1f}%, NPS {multi.get('nps', 0):.1f}% of responders")

    # 4. response propensity model (customer-grouped split)
    df = q(f"""SELECT interaction_type, channel, resolution_status, sentiment_label,
                      coalesce(customer_segment, 'unknown') customer_segment, coalesce(loyalty_tier, 'unknown') loyalty_tier,
                      CAST(csat_valid IS NOT NULL AS INT) csat_response_flag,
                      CAST(nps_valid IS NOT NULL AS INT) nps_response_flag,
                      hash(customer_id) % 100 < 20 is_test
               FROM fb_c USING SAMPLE reservoir({PROPENSITY_SAMPLE} ROWS) REPEATABLE ({SEED})""")
    X = pd.get_dummies(df[["interaction_type", "channel", "resolution_status", "sentiment_label", "customer_segment",
                           "loyalty_tier"]]).astype(float)
    res, coefs = [], []
    tr, te = ~df.is_test, df.is_test
    for target in ("csat_response_flag", "nps_response_flag"):
        m = LogisticRegression(max_iter=1000).fit(X[tr], df.loc[tr, target])
        a = roc_auc_score(df.loc[te, target], m.predict_proba(X[te])[:, 1])
        res.append({"target": target, "response_rate": round(df[target].mean(), 4), "test_auc": round(a, 4),
                    "n_train": int(tr.sum()), "n_test": int(te.sum()),
                    "interpretation": "not predictable from observables" if a < MCAR_AUC_ALERT
                                      else "PREDICTABLE - missingness depends on observables"})
        c = pd.DataFrame({"target": target, "feature": X.columns, "coef": m.coef_[0]})
        coefs.append(c.reindex(c.coef.abs().sort_values(ascending=False).index).head(10))
        if a >= MCAR_AUC_ALERT:
            flag("S7A", f"{target}: propensity AUC {a:.3f} -> survey availability is predictable; survey averages are not representative")
        else:
            flag("S7A", f"{target}: propensity AUC {a:.3f} (~0.5) -> availability not predictable from type/channel/status/sentiment/segment/tier")
    save(pd.DataFrame(res), "s7a_response_propensity_auc")
    save(pd.concat(coefs), "s7a_response_propensity_top_coefficients", show=20)

    cov = by_dim[by_dim.dimension == "interaction_type"]
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(cov))
    ax.bar(x - 0.2, cov.csat_coverage_pct, 0.4, label="CSAT coverage %"); ax.bar(x + 0.2, cov.nps_coverage_pct, 0.4, label="NPS coverage %")
    ax.set_xticks(x); ax.set_xticklabels(cov.level, rotation=30, ha="right"); ax.legend(); ax.set_title("Survey coverage by interaction type")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7a_coverage_by_interaction_type.png", dpi=120); finish_plot()


run_step(step7a_survey_missingness)

## Step 7B - Dedicated CSAT and NPS feature engineering checks (NEW)
Features are built in Section 4A (cm_feat); this step validates them and shows how they behave.

CSAT: csat_normalized = (CSAT - 1) / 4 x 100 | csat_available | csat_response_count (valid ratings) | avg_csat_rating |
latest_csat_rating | csat_health_score (mean normalised CSAT - BHS component) | csat_low_flag (any rating <= 2) |
csat_high_flag (any rating >= 4). The review row "csat_low_flag - 1 if CSAT = 4 - High satisfaction" mixes two
features, so both are provided.

NPS: availability, valid response count, average and latest rating, detractor/passive/promoter counts and shares,
nps_loyalty_score (mean of 0 / 50 / 100 category points) and nps_net_score = (promoter share - detractor share) x 100.
nps_net_score is an individual-level loyalty indicator, not the company's official NPS (population level).

In [ ]:
def nps_features_py(ratings):
    """Reference implementation used to unit-test the SQL features."""
    r = [x for x in ratings if x is not None and NPS_MIN <= x <= NPS_MAX and float(x).is_integer()]
    if not r:
        return dict(nps_response_count=0, avg_nps_rating=None, nps_net_score=None, nps_loyalty_score=None)
    cat = ["detractor" if x <= 6 else "passive" if x <= 8 else "promoter" for x in r]
    n = len(r)
    return dict(nps_response_count=n, avg_nps_rating=sum(r) / n,
                nps_net_score=(cat.count("promoter") - cat.count("detractor")) / n * 100,
                nps_loyalty_score=sum(NPS_LOYALTY_POINTS[c] for c in cat) / n)


def step7b_feedback_features():
    header("STEP 7B - CSAT and NPS feature checks")
    ex = nps_features_py([3, 9])     # worked example from the review: detractor 3 + promoter 9
    display(pd.DataFrame([{"ratings": "3, 9", **ex}]))
    assert ex["nps_net_score"] == 0 and ex["avg_nps_rating"] == 6, "NPS reference implementation failed"

    smp = q(f"""SELECT f.customer_id, f.mth, list(n.nps_valid) ratings, any_value(f.nps_net_score) net,
                       any_value(f.avg_nps_rating) avg_r, any_value(f.nps_loyalty_score) loy
                FROM (SELECT * FROM (SELECT * FROM cm_feat WHERE nps_available = 1)
                      USING SAMPLE reservoir(2000 ROWS) REPEATABLE ({SEED})) f
                JOIN fb_i n USING (customer_id, mth) WHERE n.nps_valid IS NOT NULL GROUP BY 1, 2""")
    ref = pd.DataFrame([nps_features_py(list(r)) for r in smp.ratings])
    ok = (np.allclose(ref.nps_net_score, smp.net) and np.allclose(ref.avg_nps_rating, smp.avg_r)
          and np.allclose(ref.nps_loyalty_score, smp.loy))
    flag("S7B", f"NPS SQL features match the Python reference on {len(smp):,} customer-months: {'PASS' if ok else 'FAIL'}")

    inv = q("""SELECT sum(csat_invalid_count) csat_invalid, sum(nps_invalid_count) nps_invalid,
                      sum(csat_response_count) csat_valid, sum(nps_response_count) nps_valid FROM cm_feat""")
    save(inv, "s7b_valid_vs_invalid_ratings")
    if inv.csat_invalid.iloc[0] + inv.nps_invalid.iloc[0] > 0:
        flag("S7B", "Invalid CSAT/NPS ratings found - flagged and excluded from all aggregations")

    save(q("""SELECT mth, count(*) active_cm,
                     round(100.0*avg(csat_available), 2) pct_with_csat, round(100.0*avg(nps_available), 2) pct_with_nps,
                     round(100.0*avg(CAST(csat_available = 1 AND nps_available = 1 AS INT)), 2) pct_with_both,
                     round(avg(csat_health_score), 2) avg_csat_health, round(avg(nps_net_score), 2) avg_individual_nps_net
              FROM cm_feat GROUP BY 1 ORDER BY 1"""), "s7b_feedback_coverage_by_month", show=12)

    save(q("""SELECT round(csat_health_score / 12.5) * 12.5 csat_health_bucket, count(*) n
              FROM cm_feat WHERE csat_available = 1 GROUP BY 1 ORDER BY 1"""), "s7b_csat_health_distribution", show=0)
    rel = save(q("""SELECT CAST(f.csat_valid AS INT) csat, count(*) n, round(avg(i.sscore), 3) avg_sentiment_score_gt,
                           round(100.0*count_if(i.sentiment_label = 'negative')/count(*), 2) pct_negative_label
                    FROM fb_i f JOIN itx i USING (interaction_id) WHERE f.csat_valid IS NOT NULL GROUP BY 1 ORDER BY 1"""),
               "s7b_csat_vs_sentiment")
    r = q("SELECT corr(f.csat_valid, i.sscore) r FROM fb_i f JOIN itx i USING (interaction_id) WHERE f.csat_valid IS NOT NULL").iloc[0, 0]
    flag("S7B", f"CSAT vs sentiment score: Pearson r = {r:.3f}; sentiment rises monotonically with CSAT: "
                f"{bool(rel.avg_sentiment_score_gt.is_monotonic_increasing)}")

    save(q("""SELECT nps_response_count, count(*) customer_months,
                     round(avg(avg_nps_rating), 2) avg_rating, round(avg(nps_net_score), 2) avg_net_score,
                     round(avg(nps_loyalty_score), 2) avg_loyalty_score
              FROM cm_feat WHERE nps_available = 1 GROUP BY 1 ORDER BY 1"""), "s7b_nps_response_count_profile")
    off = save(q("""SELECT mth,
                           round(100.0*(sum(nps_promoter_count) - sum(nps_detractor_count))/sum(nps_response_count), 2) official_nps_population,
                           round(avg(nps_net_score), 2) mean_individual_net_score, sum(nps_response_count) responses
                    FROM cm_feat WHERE nps_available = 1 GROUP BY 1 ORDER BY 1"""), "s7b_official_nps_vs_individual", show=12)
    flag("S7B", f"Official NPS (population) averages {off.official_nps_population.mean():.1f}; use it for reporting. "
                "Customer-month nps_net_score is an individual loyalty indicator only")

    d = q("SELECT nps_net_score, avg_nps_rating FROM cm_feat WHERE nps_available = 1")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].hist(d.nps_net_score, bins=21); ax[0].set_title("Customer-month nps_net_score")
    ax[1].scatter(d.avg_nps_rating.head(5000), d.nps_net_score.head(5000), s=4, alpha=.3)
    ax[1].set_xlabel("avg_nps_rating"); ax[1].set_ylabel("nps_net_score"); ax[1].set_title("Average rating vs net score (5k)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7b_nps_features.png", dpi=120); finish_plot()


run_step(step7b_feedback_features)

## Step 7 extension (NEW - v3): 7C interaction channel | 7D interaction type | 7E engagement channel | 7F cross-dataset
Only comparisons that are **not already produced** are added. Existing outputs that are reused, not repeated:

| Already in the notebook | Reused for | Added in v3 (missing piece) |
|---|---|---|
| s7_channel_by_type (counts) | 7C.1 | row % mix, lift vs overall mix, Cramer's V |
| s7_by_channel (negative %, avg sentiment, survey fill) | 7C.2, 7C.4 | positive / neutral %, Cramer's V |
| - | 7C.3, 7C.5 | resolution status mix, FCR rate, CSAT with vs without FCR |
| s7a_coverage_by_dimension (avg CSAT/NPS by channel / type) | 7C.4, 7D | CSAT low/high %, NPS category mix, official NPS index, Kruskal-Wallis |
| s7_sentiment_by_type (sentiment % by type) | 7D | joined into the type comparison (read from file, not recomputed) |
| s3a_channel_kpis (complaint / negative / unresolved / cancellation by channel) | 7C | headline rates are not repeated |

All CSAT/NPS statistics use valid ratings only (fb_i); nothing is imputed. sentiment_label is ground truth -> EDA/evaluation only.

In [ ]:
def channel_x(dim_sql: str, name: str, show: int = 12):
    """Row-% crosstab interaction channel x <dim>, with lift vs the all-channel mix and Cramer's V."""
    d = q(f"""SELECT CAST(channel AS VARCHAR) channel, CAST({dim_sql} AS VARCHAR) level, count(*) n
              FROM fb_i GROUP BY ALL""")
    ct = d.pivot(index="channel", columns="level", values="n").fillna(0)
    v, p = cramers_v(ct)
    pct = (100 * ct.div(ct.sum(1), axis=0)).round(2)
    lift = (pct / (100 * ct.sum(0) / ct.values.sum())).round(3)
    out = pct.add_prefix("pct_").join(lift.add_prefix("lift_")).reset_index()
    out.insert(1, "n", ct.sum(1).values)
    save(out, name, show=show)
    return out, v, p


def step7c_interaction_channel():
    header("STEP 7C - Interaction channel analysis (channel x type / sentiment / resolution / CSAT & NPS / FCR)")
    # 7C.1 channel x interaction type (counts exist in s7_channel_by_type -> add mix %, lift, association)
    mix, v, _ = channel_x("interaction_type", "s7c_channel_x_type_pct")
    lifts = mix.melt(id_vars=["channel", "n"], value_vars=[c for c in mix if c.startswith("lift_")], var_name="variable")
    top = lifts.sort_values("value", ascending=False).iloc[0]
    flag("S7C", f"Channel x interaction type: Cramer's V {v:.4f}; largest over-representation = {top.variable[5:]} via "
                f"{top.channel} (lift {top.value:.2f})")

    # 7C.2 channel x sentiment (negative % exists in s7_by_channel -> add full mix + association)
    sen, v, _ = channel_x("sentiment_label", "s7c_channel_x_sentiment")
    flag("S7C", f"Channel x sentiment: Cramer's V {v:.4f} "
                f"({'material' if v >= CHANNEL_EFFECT_MIN else 'negligible'} association)")

    # 7C.3 channel x resolution status
    res, v, _ = channel_x("resolution_status", "s7c_channel_x_resolution")
    flag("S7C", f"Channel x resolution status: Cramer's V {v:.4f}; lowest resolved share = "
                f"{res.loc[res.pct_resolved.idxmin(), 'channel'] if 'pct_resolved' in res else 'n/a'}")

    # 7C.4 channel x CSAT and NPS (distribution + official NPS index; averages already in s7a_coverage_by_dimension)
    from scipy.stats import kruskal
    sv = save(q(f"""SELECT CAST(channel AS VARCHAR) channel,
                           count(csat_valid) n_csat,
                           round(100.0*count_if(csat_valid <= 2)/nullif(count(csat_valid), 0), 2) pct_csat_low,
                           round(100.0*count_if(csat_valid = 3)/nullif(count(csat_valid), 0), 2) pct_csat_neutral,
                           round(100.0*count_if(csat_valid >= 4)/nullif(count(csat_valid), 0), 2) pct_csat_high,
                           count(nps_valid) n_nps,
                           round(100.0*count_if(nps_category = 'promoter')/nullif(count(nps_valid), 0), 2) pct_promoter,
                           round(100.0*count_if(nps_category = 'passive')/nullif(count(nps_valid), 0), 2) pct_passive,
                           round(100.0*count_if(nps_category = 'detractor')/nullif(count(nps_valid), 0), 2) pct_detractor,
                           CASE WHEN count(nps_valid) >= 100 THEN round(100.0*(count_if(nps_category = 'promoter')
                                - count_if(nps_category = 'detractor'))/count(nps_valid), 1) END official_nps_index
                    FROM fb_i GROUP BY 1 ORDER BY official_nps_index"""), "s7c_channel_x_csat_nps")
    tests = []
    for col in ("csat_valid", "nps_valid"):
        d = q(f"SELECT CAST(channel AS VARCHAR) channel, {col} v FROM fb_i WHERE {col} IS NOT NULL")
        g = [x.v.to_numpy() for _, x in d.groupby("channel")]
        h, p = kruskal(*g)
        tests.append({"metric": col.replace("_valid", ""), "kruskal_h": round(h, 2), "p_value": p,
                      "epsilon_squared": round((h - len(g) + 1) / (len(d) - len(g)), 5)})
    tests = save(pd.DataFrame(tests), "s7c_channel_csat_nps_tests")
    flag("S7C", f"Channel x CSAT/NPS: NPS index ranges {sv.official_nps_index.min()} to {sv.official_nps_index.max()}; "
                f"effect size CSAT {tests.epsilon_squared.iloc[0]}, NPS {tests.epsilon_squared.iloc[1]} "
                f"({'material' if tests.epsilon_squared.max() >= CHANNEL_EFFECT_MIN else 'negligible'})")

    # 7C.5 channel x first contact resolution
    fcr = save(q(f"""SELECT CAST(channel AS VARCHAR) channel, count(*) n,
                            round(100.0*count_if(fcr)/count(*), 2) fcr_rate,
                            round(100.0*count_if(fcr AND interaction_type IN {sql_list(COMPLAINT_TYPES)})
                                  /nullif(count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)}), 0), 2) fcr_rate_complaints,
                            round(avg(csat_valid) FILTER (WHERE fcr), 3) avg_csat_fcr,
                            round(avg(csat_valid) FILTER (WHERE NOT fcr), 3) avg_csat_no_fcr,
                            round(avg(nps_valid) FILTER (WHERE fcr), 3) avg_nps_fcr,
                            round(avg(nps_valid) FILTER (WHERE NOT fcr), 3) avg_nps_no_fcr
                     FROM fb_i GROUP BY 1 ORDER BY fcr_rate"""), "s7c_channel_x_fcr")
    fcr["csat_gain_from_fcr"] = fcr.avg_csat_fcr - fcr.avg_csat_no_fcr
    flag("S7C", f"Channel x FCR: FCR rate {fcr.fcr_rate.min()}% ({fcr.channel.iloc[0]}) to {fcr.fcr_rate.max()}% "
                f"({fcr.channel.iloc[-1]}); CSAT gain when resolved first time {fcr.csat_gain_from_fcr.mean():+.2f} on average")

    heat = mix.set_index("channel")[[c for c in mix if c.startswith("pct_")]]
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
    im = ax[0].imshow(heat.values, cmap="Blues", aspect="auto")
    ax[0].set_xticks(range(heat.shape[1])); ax[0].set_xticklabels([c[4:] for c in heat], rotation=40, ha="right", fontsize=8)
    ax[0].set_yticks(range(heat.shape[0])); ax[0].set_yticklabels(heat.index)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax[0].text(j, i, f"{heat.values[i, j]:.0f}", ha="center", va="center", fontsize=7)
    ax[0].set_title("Interaction type mix by channel (row %)"); plt.colorbar(im, ax=ax[0])
    ax[1].bar(fcr.channel, fcr.fcr_rate, color="#00B050"); ax[1].tick_params(axis="x", rotation=30)
    ax[1].set_title("First contact resolution rate by channel (%)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7c_interaction_channel.png", dpi=120); finish_plot()


run_step(step7c_interaction_channel)

### 7D. Interaction type analysis (NEW - v3)
One comparison table across all interaction types (sentiment, CSAT, NPS, resolution, FCR), then the requested
profiles for technical_issue, feedback, product_request and compliment. Sentiment % by type is read from the existing
s7_sentiment_by_type output instead of being recomputed.

In [ ]:
def step7d_interaction_type():
    header("STEP 7D - Interaction type analysis")
    from scipy.stats import kruskal
    kpi = q(f"""SELECT CAST(interaction_type AS VARCHAR) interaction_type, count(*) n_total,
                       round(100.0*count_if(resolution_status IN {sql_list(RESOLVED_STATUS)})/count(*), 2) resolution_rate,
                       round(100.0*count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)})/count(*), 2) unresolved_rate,
                       round(100.0*count_if(fcr)/count(*), 2) fcr_rate,
                       round(100.0*count_if(resolution_status = 'resolved')/count(*), 2) pct_resolved,
                       round(100.0*count_if(resolution_status = 'escalated')/count(*), 2) pct_escalated,
                       round(100.0*count_if(resolution_status = 'pending')/count(*), 2) pct_pending,
                       round(100.0*count_if(resolution_status = 'unresolved')/count(*), 2) pct_unresolved,
                       count(csat_valid) n_csat, round(avg(csat_valid), 3) avg_csat,
                       round(100.0*count_if(csat_valid <= 2)/nullif(count(csat_valid), 0), 2) pct_csat_low,
                       count(nps_valid) n_nps, round(avg(nps_valid), 3) avg_nps,
                       CASE WHEN count(nps_valid) >= 100 THEN round(100.0*(count_if(nps_category = 'promoter')
                            - count_if(nps_category = 'detractor'))/count(nps_valid), 1) END nps_index
                FROM fb_i GROUP BY 1""")
    sent_path = OUT_DIR / "s7_sentiment_by_type.csv"           # produced by Step 7 - reuse, do not recompute
    if sent_path.exists():
        sent = pd.read_csv(sent_path).drop(columns=["n"], errors="ignore")
    else:
        sent = q("""SELECT interaction_type, round(100.0*count_if(sentiment_label='positive')/count(*), 2) pct_positive,
                           round(100.0*count_if(sentiment_label='neutral')/count(*), 2) pct_neutral,
                           round(100.0*count_if(sentiment_label='negative')/count(*), 2) pct_negative,
                           round(avg(sscore), 3) avg_sentiment_gt FROM itx WHERE in_window GROUP BY 1""")
    comp = kpi.merge(sent, on="interaction_type", how="left").sort_values("avg_csat")
    overall = q(f"""SELECT round(100.0*count_if(resolution_status IN {sql_list(RESOLVED_STATUS)})/count(*), 2) resolution_rate,
                           round(100.0*count_if(fcr)/count(*), 2) fcr_rate, round(avg(csat_valid), 3) avg_csat,
                           round(avg(nps_valid), 3) avg_nps FROM fb_i""").iloc[0]
    for m in ("resolution_rate", "fcr_rate", "avg_csat", "avg_nps"):
        comp[f"{m}_vs_overall"] = (comp[m] - overall[m]).round(3)
    comp = save(comp, "s7d_interaction_type_comparison", show=10)

    # requested profiles (subset of the comparison + gap to the all-type average)
    rows = []
    for t, metrics in TYPE_PROFILES.items():
        r = comp[comp.interaction_type == t]
        if r.empty:
            flag("S7D", f"Interaction type '{t}' not found in data - profile skipped (check TYPE_PROFILES)")
            continue
        for m in metrics:
            all_types = comp[m].astype(float)
            rows.append({"interaction_type": t, "metric": m, "value": r[m].iloc[0],
                         "all_type_mean": round(all_types.mean(), 3), "rank_high_to_low": int(all_types.rank(ascending=False)[r.index[0]]),
                         "n_types": len(comp)})
    prof = save(pd.DataFrame(rows), "s7d_interaction_type_profiles", show=40)
    for t in TYPE_PROFILES:
        p = prof[prof.interaction_type == t].set_index("metric").value
        if p.empty:
            continue
        flag("S7D", f"{t}: " + ", ".join(f"{m}={p[m]}" for m in p.index))

    tests = []
    for col in ("csat_valid", "nps_valid"):
        d = q(f"SELECT CAST(interaction_type AS VARCHAR) t, {col} v FROM fb_i WHERE {col} IS NOT NULL")
        g = [x.v.to_numpy() for _, x in d.groupby("t")]
        h, p = kruskal(*g)
        tests.append({"metric": col.replace("_valid", ""), "kruskal_h": round(h, 2), "p_value": p,
                      "epsilon_squared": round((h - len(g) + 1) / (len(d) - len(g)), 4)})
    ct = q("SELECT CAST(interaction_type AS VARCHAR) t, CAST(resolution_status AS VARCHAR) s, count(*) n FROM fb_i GROUP BY ALL") \
        .pivot(index="t", columns="s", values="n").fillna(0)
    v, p = cramers_v(ct)
    tests.append({"metric": "resolution_status (Cramer's V)", "kruskal_h": None, "p_value": p, "epsilon_squared": round(v, 4)})
    tests = save(pd.DataFrame(tests), "s7d_interaction_type_tests")
    flag("S7D", f"Across interaction types: CSAT epsilon-squared {tests.epsilon_squared.iloc[0]}, NPS {tests.epsilon_squared.iloc[1]}, "
                f"resolution Cramer's V {tests.epsilon_squared.iloc[2]}; lowest CSAT = {comp.interaction_type.iloc[0]} "
                f"({comp.avg_csat.iloc[0]}), highest = {comp.interaction_type.iloc[-1]} ({comp.avg_csat.iloc[-1]})")

    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    x = np.arange(len(comp))
    ax[0].bar(x, comp.pct_negative, label="negative %", color="#C00000")
    ax[0].bar(x, comp.pct_neutral, bottom=comp.pct_negative, label="neutral %", color="#BFBFBF")
    ax[0].bar(x, comp.pct_positive, bottom=comp.pct_negative + comp.pct_neutral, label="positive %", color="#00B050")
    ax[0].set_title("Sentiment mix by interaction type"); ax[0].legend(fontsize=8)
    ax[1].bar(x - 0.2, comp.avg_csat, 0.4, label="avg CSAT (1-5)"); ax2 = ax[1].twinx()
    ax2.bar(x + 0.2, comp.avg_nps, 0.4, color="#FFC000", label="avg NPS (0-10)")
    ax[1].set_title("CSAT (left) and NPS (right) by interaction type")
    ax[2].bar(x - 0.2, comp.resolution_rate, 0.4, label="resolution %"); ax[2].bar(x + 0.2, comp.fcr_rate, 0.4, label="FCR %")
    ax[2].set_title("Resolution and FCR rate by interaction type"); ax[2].legend(fontsize=8)
    for a in ax:
        a.set_xticks(x); a.set_xticklabels(comp.interaction_type, rotation=40, ha="right", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7d_interaction_type.png", dpi=120); finish_plot()


run_step(step7d_interaction_type)

### 7E. Engagement channel analysis (NEW - v3)
Uses the verified engagement groups (ENG_POSITIVE / ENG_NEUTRAL / ENG_NEGATIVE, UNSUBSCRIBE_EVENTS).
Push-notification relationship: engagement quality is measured on **non-push** events only, so push opens/dismissals
cannot mechanically drive the result.

In [ ]:
def step7e_engagement_channel():
    header("STEP 7E - Engagement channel analysis")
    # 7E.1 channel-wise event frequency (channel totals exist in 3A -> add channel x event mix)
    d = q("""SELECT CAST(engagement_channel AS VARCHAR) engagement_channel, CAST(event_type AS VARCHAR) event_type, count(*) n
             FROM eng WHERE in_window GROUP BY ALL""")
    d["pct_of_channel"] = (100 * d.n / d.groupby("engagement_channel").n.transform("sum")).round(2)
    save(d.sort_values(["engagement_channel", "n"], ascending=[True, False]), "s7e_engagement_channel_x_event", show=0)
    piv = d.pivot(index="engagement_channel", columns="event_type", values="pct_of_channel").fillna(0)
    display(piv.round(1))
    single = (piv > 0).sum(1)
    flag("S7E", "Events per engagement channel: " + ", ".join(f"{c}={n}" for c, n in single.items()))

    # 7E.2-7E.5 channel x engagement polarity, conversion, unsubscribe/opt-out, session duration
    k = save(q(f"""SELECT CAST(engagement_channel AS VARCHAR) engagement_channel, count(*) n_events,
                          count(DISTINCT customer_id) n_customers,
                          round(100.0*count_if(event_type IN {sql_list(ENG_POSITIVE)})/count(*), 2) pct_positive,
                          round(100.0*count_if(event_type IN {sql_list(ENG_NEUTRAL)})/count(*), 2) pct_neutral,
                          round(100.0*count_if(event_type IN {sql_list(ENG_NEGATIVE)})/count(*), 2) pct_negative,
                          count_if(converted) n_conversions,
                          round(100.0*count_if(converted)/count(*), 3) conversion_rate,
                          count_if(event_type IN {sql_list(UNSUBSCRIBE_EVENTS)}) n_unsubscribe_optout,
                          round(100.0*count_if(event_type IN {sql_list(UNSUBSCRIBE_EVENTS)})/count(*), 3) unsubscribe_optout_rate,
                          round(100.0*count(DISTINCT customer_id) FILTER (WHERE event_type IN {sql_list(UNSUBSCRIBE_EVENTS)})
                                /count(DISTINCT customer_id), 3) pct_customers_unsubscribed,
                          round(100.0*count_if(dur = 0 OR dur IS NULL)/count(*), 2) pct_zero_duration,
                          round(median(dur) FILTER (WHERE dur > 0), 1) median_session_sec,
                          round(avg(dur) FILTER (WHERE dur > 0), 1) mean_session_sec,
                          round(quantile_cont(dur, 0.9) FILTER (WHERE dur > 0), 1) p90_session_sec
                   FROM eng WHERE in_window GROUP BY 1 ORDER BY n_events DESC"""), "s7e_engagement_channel_kpis", show=10)
    ct = q(f"""SELECT CAST(engagement_channel AS VARCHAR) c,
                      CASE WHEN event_type IN {sql_list(ENG_POSITIVE)} THEN 'positive' WHEN event_type IN {sql_list(ENG_NEGATIVE)}
                           THEN 'negative' ELSE 'neutral' END pol, count(*) n FROM eng WHERE in_window GROUP BY ALL""") \
        .pivot(index="c", columns="pol", values="n").fillna(0)
    v, _ = cramers_v(ct)
    flag("S7E", f"Engagement polarity by channel (Cramer's V {v:.3f}): most negative = "
                f"{k.loc[k.pct_negative.idxmax(), 'engagement_channel']} ({k.pct_negative.max()}%), most positive = "
                f"{k.loc[k.pct_positive.idxmax(), 'engagement_channel']} ({k.pct_positive.max()}%)")
    flag("S7E", f"Conversion: highest {k.loc[k.conversion_rate.idxmax(), 'engagement_channel']} ({k.conversion_rate.max()}%); "
                f"unsubscribe/opt-out highest {k.loc[k.unsubscribe_optout_rate.idxmax(), 'engagement_channel']} "
                f"({k.unsubscribe_optout_rate.max()}%)")
    flag("S7E", f"Session duration (non-zero sessions; channels without durations shown as NA): longest median {k.loc[k.median_session_sec.idxmax(), 'engagement_channel']} "
                f"({k.median_session_sec.max()}s); zero-duration share up to {k.pct_zero_duration.max()}% "
                "-> do not compare duration across channels without excluding zero/NA sessions")

    # 7E.6 push notification usage vs engagement quality (customer-month level)
    pm = q(f"""
        WITH m AS (
            SELECT customer_id, mth,
                   count_if(engagement_channel = '{PUSH_CHANNEL}') push_events,
                   count_if(event_type IN {sql_list(PUSH_OPEN_EVENTS)}) push_opens,
                   count_if(event_type IN {sql_list(PUSH_DISMISS_EVENTS)}) push_dismisses,
                   count_if(engagement_channel <> '{PUSH_CHANNEL}' AND event_type IN {sql_list(ENG_POSITIVE)}) np_pos,
                   count_if(engagement_channel <> '{PUSH_CHANNEL}' AND event_type IN {sql_list(ENG_NEGATIVE)}) np_neg,
                   count_if(engagement_channel <> '{PUSH_CHANNEL}') np_events,
                   count_if(converted) conversions,
                   count_if(event_type IN {sql_list(UNSUBSCRIBE_EVENTS)}) unsubscribes
            FROM eng WHERE in_window GROUP BY ALL)
        SELECT *,
               CASE WHEN push_events = 0 THEN '0_no_push'
                    WHEN push_opens + push_dismisses = 0 THEN '1_push_other_only'
                    WHEN push_opens * 1.0 / (push_opens + push_dismisses) >= 0.5 THEN '3_mostly_opens'
                    ELSE '2_mostly_dismisses' END push_usage,
               push_opens * 1.0 / nullif(push_opens + push_dismisses, 0) push_open_rate,
               np_pos * 1.0 / nullif(np_pos + np_neg, 0) non_push_quality
        FROM m""")
    pu = pm.groupby("push_usage").agg(customer_months=("customer_id", "size"),
                                      avg_non_push_events=("np_events", "mean"),
                                      avg_non_push_quality=("non_push_quality", "mean"),
                                      pct_with_conversion=("conversions", lambda s: 100 * (s > 0).mean()),
                                      pct_with_unsubscribe=("unsubscribes", lambda s: 100 * (s > 0).mean())).round(4).reset_index()
    save(pu, "s7e_push_usage_vs_engagement_quality")
    m = pm.push_open_rate.notna() & pm.non_push_quality.notna()
    rho = spearmanr(pm.loc[m, "push_open_rate"], pm.loc[m, "non_push_quality"])[0] if m.sum() > 30 else np.nan
    save(pd.DataFrame([{"pairs": int(m.sum()), "spearman_push_open_rate_vs_non_push_quality": rho}]), "s7e_push_quality_correlation")
    flag("S7E", f"Push usage vs engagement quality: Spearman(push open rate, non-push quality) = {rho:.3f} on {int(m.sum()):,} "
                f"customer-months; non-push quality {pu.avg_non_push_quality.min():.3f}-{pu.avg_non_push_quality.max():.3f} across push groups")
    del pm
    gc.collect()

    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    x = np.arange(len(k))
    ax[0].bar(x, k.pct_negative, color="#C00000", label="negative %")
    ax[0].bar(x, k.pct_neutral, bottom=k.pct_negative, color="#BFBFBF", label="neutral %")
    ax[0].bar(x, k.pct_positive, bottom=k.pct_negative + k.pct_neutral, color="#00B050", label="positive %")
    ax[0].set_title("Engagement polarity by channel"); ax[0].legend(fontsize=8)
    ax[1].bar(x - 0.2, k.conversion_rate, 0.4, label="conversion %"); ax[1].bar(x + 0.2, k.unsubscribe_optout_rate, 0.4, label="unsub/opt-out %")
    ax[1].set_title("Conversion vs unsubscribe/opt-out"); ax[1].legend(fontsize=8)
    ax[2].bar(x, k.median_session_sec, color="#4472C4"); ax[2].set_title("Median session duration (s, non-zero)")
    for a in ax:
        a.set_xticks(x); a.set_xticklabels(k.engagement_channel, rotation=30, ha="right", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7e_engagement_channel.png", dpi=120); finish_plot()


run_step(step7e_engagement_channel)

### 7F. Cross-dataset analysis (NEW - v3)
Interactions x engagements at **customer-month** level (cm_feat for 7F.2-7F.4; months with both sources only).
Results are associations, not causal effects. CSAT uses valid ratings only; months without CSAT are excluded, not filled.

In [ ]:
def step7f_cross_dataset():
    header("STEP 7F - Cross-dataset analysis: interactions x engagements")
    # 7F.1 dominant interaction channel vs dominant engagement channel (same customer-month)
    x = q("""WITH i AS (SELECT customer_id, mth, mode(CAST(channel AS VARCHAR)) int_channel FROM itx WHERE in_window GROUP BY ALL),
                  e AS (SELECT customer_id, mth, mode(CAST(engagement_channel AS VARCHAR)) eng_channel FROM eng WHERE in_window GROUP BY ALL)
             SELECT int_channel, eng_channel, count(*) n FROM i JOIN e USING (customer_id, mth) GROUP BY ALL""")
    ct = x.pivot(index="int_channel", columns="eng_channel", values="n").fillna(0)
    v, p = cramers_v(ct)
    pct = (100 * ct.div(ct.sum(1), axis=0)).round(2)
    lift = (pct / (100 * ct.sum(0) / ct.values.sum())).round(3)
    save(pct.add_prefix("pct_").join(lift.add_prefix("lift_")).reset_index(), "s7f_interaction_vs_engagement_channel")
    flag("S7F", f"Interaction channel vs engagement channel ({int(ct.values.sum()):,} customer-months): Cramer's V {v:.4f} "
                f"({'channels are linked' if v >= CHANNEL_EFFECT_MIN else 'no meaningful link - channel preference differs by purpose'})")

    both = q("""SELECT int_count, complaint_count, sentiment_neg_share, csat_available, avg_csat_rating,
                       n_eng, eng_negative_count, positive_engagement_share,
                       eng_negative_count * 1.0 / nullif(n_eng, 0) negative_engagement_share
                FROM cm_feat WHERE has_interactions = 1 AND has_engagements = 1""")
    res = []

    # 7F.2 negative sentiment vs negative engagement
    both["neg_sent_band"] = pd.cut(both.sentiment_neg_share, [-0.001, 0, 0.5, 1], labels=["0_none", "1_some (<=50%)", "2_mostly (>50%)"])
    t = both.groupby("neg_sent_band", observed=True).agg(customer_months=("n_eng", "size"),
                                                          avg_negative_engagement_share=("negative_engagement_share", "mean"),
                                                          pct_with_negative_engagement=("eng_negative_count", lambda s: 100 * (s > 0).mean())).round(4).reset_index()
    save(t, "s7f_negative_sentiment_vs_negative_engagement")
    res.append(("negative sentiment share", "negative engagement share",
                spearmanr(both.sentiment_neg_share, both.negative_engagement_share, nan_policy="omit")[0], len(both)))

    # 7F.3 complaint frequency vs engagement quality
    both["complaint_band"] = pd.cut(both.complaint_count, [-1, 0, 1, np.inf], labels=["0", "1", "2+"])
    t = both.groupby("complaint_band", observed=True).agg(customer_months=("n_eng", "size"),
                                                           avg_positive_engagement_share=("positive_engagement_share", "mean"),
                                                           avg_engagement_events=("n_eng", "mean")).round(4).reset_index()
    save(t, "s7f_complaints_vs_engagement_quality")
    res.append(("complaint count", "positive engagement share",
                spearmanr(both.complaint_count, both.positive_engagement_share, nan_policy="omit")[0],
                int(both.positive_engagement_share.notna().sum())))

    # 7F.4 CSAT vs negative engagement (valid CSAT months only)
    c = both[both.csat_available == 1].copy()
    c["csat_band"] = c.avg_csat_rating.round().astype(int)
    t = c.groupby("csat_band").agg(customer_months=("n_eng", "size"),
                                   avg_negative_engagement_share=("negative_engagement_share", "mean"),
                                   pct_with_negative_engagement=("eng_negative_count", lambda s: 100 * (s > 0).mean())).round(4).reset_index()
    save(t, "s7f_csat_vs_negative_engagement")
    res.append(("avg CSAT", "negative engagement share",
                spearmanr(c.avg_csat_rating, c.negative_engagement_share, nan_policy="omit")[0], len(c)))

    r = save(pd.DataFrame(res, columns=["interaction_metric", "engagement_metric", "spearman", "customer_months"]).round(4),
             "s7f_cross_dataset_correlations")
    for row in r.itertuples():
        flag("S7F", f"{row.interaction_metric} vs {row.engagement_metric}: Spearman {row.spearman:.3f} "
                    f"({'meaningful' if abs(row.spearman) >= 0.10 else 'weak/none'}) on {row.customer_months:,} customer-months")
    del both, c
    gc.collect()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    im = ax.imshow(lift.values, cmap="RdBu_r", vmin=0.5, vmax=1.5, aspect="auto")
    ax.set_xticks(range(lift.shape[1])); ax.set_xticklabels(lift.columns, rotation=30, ha="right")
    ax.set_yticks(range(lift.shape[0])); ax.set_yticklabels(lift.index)
    ax.set_title("Lift: dominant interaction channel (rows) x engagement channel (cols)"); plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7f_interaction_vs_engagement_channel.png", dpi=120); finish_plot()


run_step(step7f_cross_dataset)

## Step 8 - Relationships

In [ ]:
FEATURES = ["n_tx", "n_tx_failed", "n_refunds", "spend_pctile", "n_eng", "n_logins", "eng_seconds",
            "pages_viewed", "n_conversions", "n_int", "n_complaint", "n_negative", "n_open_issue",
            "avg_sentiment_gt", "avg_csat", "avg_nps"]


def step8_relationships():
    header("STEP 8 - Correlations and leading indicators (proxy labels)")
    cols = ", ".join(f"coalesce({c}, 0) AS {c}" if c not in ("spend_pctile", "avg_sentiment_gt", "avg_csat", "avg_nps")
                     else c for c in FEATURES)
    df = q(f"""SELECT {cols} FROM (SELECT * FROM cm WHERE active)
               USING SAMPLE reservoir({CORR_SAMPLE_ROWS} ROWS) REPEATABLE (42)""")
    corr = df.corr(method="spearman").round(3)
    save(corr.reset_index().rename(columns={"index": "feature"}), "s8_spearman_correlation", show=0)
    pairs = []
    for i, a in enumerate(FEATURES):
        for b in FEATURES[i + 1:]:
            r = corr.loc[a, b]
            if pd.notna(r) and abs(r) >= HIGH_CORR:
                pairs.append({"feature_a": a, "feature_b": b, "spearman": r})
                flag("S8", f"{a} ~ {b} (rho={r}) -> redundant; keep one")
    save(pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman"]), "s8_high_corr_pairs")

    # Do complaints / negative / cancellation requests precede inactivity or decline?
    lead = save(q(f"""
        WITH x AS (
            SELECT customer_id, mth, active, n_int, n_complaint, n_negative, n_cancel_req,
                   coalesce(gross_spend, 0) spend, coalesce(n_eng, 0) eng,
                   lead(CAST(active AS INT), 1) OVER w active_next1,
                   max(CAST(active AS INT)) OVER (PARTITION BY customer_id ORDER BY mth
                                                  ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING) active_next3,
                   coalesce(lead(gross_spend, 1) OVER w, 0) spend_next1,
                   coalesce(lead(n_eng, 1) OVER w, 0) eng_next1
            FROM cm WINDOW w AS (PARTITION BY customer_id ORDER BY mth))
        SELECT CASE WHEN n_cancel_req > 0 THEN '1_cancellation_request'
                    WHEN n_complaint > 0 THEN '2_complaint_or_dispute'
                    WHEN n_negative > 0 THEN '3_negative_interaction'
                    WHEN coalesce(n_int, 0) > 0 THEN '4_other_interaction'
                    ELSE '5_no_interaction' END signal_in_month_M,
               count(*) n_customer_months,
               round(100.0*avg(active_next1), 2) pct_active_M1,
               round(100.0*(1 - avg(active_next3)), 2) pct_inactive_next_3_months,
               round(100.0*count_if(spend > 0 AND spend_next1 < 0.7*spend)/nullif(count_if(spend > 0), 0), 2)
                   pct_spend_drop_gt30_M1,
               round(100.0*count_if(eng > 0 AND eng_next1 < 0.7*eng)/nullif(count_if(eng > 0), 0), 2)
                   pct_eng_drop_gt30_M1
        FROM x
        WHERE active AND mth <= DATE '{WINDOW_END}' - INTERVAL 4 MONTH
        GROUP BY 1 ORDER BY 1"""), "s8_signal_vs_next_months")
    base = lead[lead.signal_in_month_M == "5_no_interaction"].pct_inactive_next_3_months
    canc = lead[lead.signal_in_month_M == "1_cancellation_request"].pct_inactive_next_3_months
    if len(base) and len(canc):
        lift = canc.iloc[0] / max(base.iloc[0], 1e-9)
        flag("S8", f"Cancellation request -> inactive next 3 months: {canc.iloc[0]:.1f}% vs {base.iloc[0]:.1f}% "
                   f"baseline (lift {lift:.2f}x). {'Usable' if lift >= 1.3 else 'Weak'} as churn proxy label")


run_step(step8_relationships)

## Step 9 - Segments (context only)

In [ ]:
SEGMENT_DIMS = ["customer_segment", "loyalty_tier", "region", "preferred_channel", "tenure_band", "age_band"]


def segment_view():
    con.execute("""
        CREATE OR REPLACE TEMP VIEW cm_seg AS
        SELECT m.*, c.customer_segment, c.loyalty_tier, c.region, c.preferred_channel,
               CASE WHEN c.tenure_n < 12 THEN '0-11m' WHEN c.tenure_n < 24 THEN '12-23m'
                    WHEN c.tenure_n < 48 THEN '24-47m' WHEN c.tenure_n IS NULL THEN 'unknown' ELSE '48m+' END tenure_band,
               CASE WHEN c.age_n < 25 THEN '<25' WHEN c.age_n < 35 THEN '25-34' WHEN c.age_n < 50 THEN '35-49'
                    WHEN c.age_n < 65 THEN '50-64' WHEN c.age_n IS NULL THEN 'unknown' ELSE '65+' END age_band
        FROM cm m JOIN cust1 c USING (customer_id)""")


def step9_segments():
    header("STEP 9 - Segment behaviour (decide global vs segment-relative scoring)")
    segment_view()
    metrics = {
        "pct_active": "100.0*avg(CAST(active AS INT))",
        "avg_tx_when_active": "avg(n_tx) FILTER (WHERE active)",
        "avg_spend_pctile": "avg(spend_pctile)",
        "avg_eng_when_active": "avg(coalesce(n_eng,0)) FILTER (WHERE active)",
        "pct_with_interaction": "100.0*count_if(has_int)/nullif(count_if(active),0)",
        "complaint_rate": "100.0*count_if(n_complaint > 0)/nullif(count_if(has_int),0)",
        "negative_rate": "100.0*count_if(n_negative > 0)/nullif(count_if(has_int),0)",
        "avg_sentiment_gt": "avg(avg_sentiment_gt)",
        "pct_2plus_sources": "100.0*count_if(n_sources >= 2)/nullif(count_if(active),0)",
    }
    sel = ", ".join(f"round({v}, 3) AS {k}" for k, v in metrics.items())
    frames, spread = [], []
    for dim in SEGMENT_DIMS:
        d = q(f"""SELECT '{dim}' dimension, CAST({dim} AS VARCHAR) segment,
                         count(DISTINCT customer_id) n_customers, {sel}
                  FROM cm_seg WHERE NOT pre_signup GROUP BY ALL ORDER BY n_customers DESC""")
        frames.append(d)
        for m in metrics:
            vals = d[m].dropna()
            vals = vals[vals > 0]
            if len(vals) > 1:
                ratio = vals.max() / vals.min()
                spread.append({"dimension": dim, "metric": m, "max_min_ratio": round(ratio, 3)})
                if ratio > SEGMENT_SPREAD_ALERT and m not in ("avg_sentiment_gt",):
                    flag("S9", f"{m} differs by {ratio:.2f}x across {dim} -> score vs segment/own history")
    save(pd.concat(frames), "s9_segment_profile", show=20)
    save(pd.DataFrame(spread).sort_values("max_min_ratio", ascending=False), "s9_segment_spread", show=15)


run_step(step9_segments)

## Step 10 - Leakage & eligibility

In [ ]:
COLUMN_ROLES = {   # column -> (role, note); everything else defaults to 'feature candidate'
    "customer_id": ("id", "join key"), "transaction_id": ("id", ""), "engagement_id": ("id", ""),
    "interaction_id": ("id", ""), "agent_id": ("id", "not a customer feature"),
    "sentiment_label": ("label / evaluation only", "ground truth - never a feature"),
    "sentiment_score_gt": ("label / evaluation only", "ground truth - never a feature"),
    "csat_score": ("validation label", "use to validate sentiment; sparse"),
    "nps_score": ("validation label", "use to validate sentiment; sparse"),
    "tenure_months": ("context (snapshot)", "likely as-of a single date - recompute per month from signup_date"),
    "loyalty_tier": ("context (snapshot)", "may reflect end-of-period status - context only, never a penalty"),
    "age": ("context only", "do not lower health/sentiment"), "gender": ("context only", "do not lower health/sentiment"),
    "region": ("context / segmentation", ""), "country": ("context / segmentation", ""),
    "customer_segment": ("context / segmentation", ""), "income_band": ("context only", "do not lower health/sentiment"),
    "acquisition_channel": ("context", ""), "preferred_channel": ("context / action routing", ""),
    "marketing_consent": ("action constraint", "check before recommending outreach"),
    "month": ("time key", "re-derive from timestamp"), "signup_date": ("context", "used to derive tenure per month"),
}


def step10_leakage_eligibility():
    header("STEP 10 - Leakage register and eligibility funnel")
    rows = []
    for t in TABLES:
        for c in columns_of(f"raw_{t}"):
            role, note = COLUMN_ROLES.get(c, ("feature candidate", "aggregate to customer-month using data <= month M"))
            rows.append({"table": t, "column": c, "role": role, "note": note})
    save(pd.DataFrame(rows), "s10_column_role_register", show=0)
    flag("S10", "sentiment_label / sentiment_score_gt are ground truth -> evaluation only (label leakage otherwise)")

    # Is tenure_months a snapshot? implied reference date = signup + tenure_months
    ref = save(q("""SELECT CAST(date_trunc('month', signup + to_months(CAST(tenure_n AS INT))) AS DATE) implied_as_of_month,
                           count(*) n FROM cust WHERE signup IS NOT NULL AND tenure_n IS NOT NULL
                    GROUP BY 1 ORDER BY n DESC LIMIT 10"""), "s10_tenure_implied_as_of_date")
    if len(ref):
        top = pd.Timestamp(ref.implied_as_of_month.iloc[0])
        share = 100 * ref.n.iloc[0] / q("SELECT count(*) FROM cust WHERE signup IS NOT NULL").iloc[0, 0]
        flag("S10", f"tenure_months mostly implies as-of {top:%Y-%m} ({share:.0f}% of customers) -> "
                    "snapshot field; recompute tenure for each scoring month")

    # Future-data check: features for month M must only use events with ts < end of M
    save(q("""SELECT 'transactions' src, count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) misassigned FROM tx
              UNION ALL SELECT 'engagements', count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) FROM eng
              UNION ALL SELECT 'interactions', count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) FROM itx"""),
         "s10_month_assignment_check", show=0)

    # Eligibility funnel per month
    funnel = save(q(f"""
        SELECT mth,
               count(*) all_customers,
               count_if(NOT pre_signup) signed_up,
               count_if(active) active_in_month,
               count_if({HISTORY_COND}) active_with_{MIN_HISTORY_MONTHS}m_history,
               count_if({HISTORY_COND} AND n_sources >= 2) plus_2plus_sources,
               count_if({CORE_COND}) core_tx_plus_1,
               count_if({HISTORY_COND} AND n_sources = 1) low_confidence_1_source,
               count_if(active AND months_observed < {MIN_HISTORY_MONTHS}) new_or_short_history,
               count_if(lifecycle_status = 'dormant') dormant,
               count_if(lifecycle_status = 'churn_proxy') churn_proxy
        FROM cm GROUP BY 1 ORDER BY 1"""), "s10_eligibility_funnel_by_month", show=12)
    first_ok = funnel[funnel[f"active_with_{MIN_HISTORY_MONTHS}m_history"] > 0].mth.min()
    flag("S10", f"First month with scorable history: {pd.Timestamp(first_ok):%Y-%m} "
                f"(first {MIN_HISTORY_MONTHS - 1} months are warm-up only)")


run_step(step10_leakage_eligibility)

## Step 11 - Behavioral Health Score (BHS) calibration (NEW)
| Sub-step | What it does |
|---|---|
| 11.1 | Component definitions, initial weights, normalisation to 0-100 |
| 11.2 | BHS with dynamic weight redistribution + separate Sentiment and Loyalty scores |
| 11.3 | Score coverage and component-level coverage & confidence |
| 11.4 | Confidence calculation and bands |
| 11.5 | Weight sensitivity (+/-50%, drop-one, equal, random weights) |
| 11.6 | Independent validation vs CSAT (leave-CSAT-out) and NPS (not in BHS) |
| 11.7 | Out-of-time validation vs future outcomes + data-driven weight suggestion |

BHS = sum(w_k * c_k over observed k) / sum(w_k over observed k), c_k in [0, 100];
published only if sum(w_k over observed k) >= MIN_BHS_WEIGHT_COVERAGE.

Rules: no imputation - a missing component drops out and the remaining weights are re-normalised; profile attributes
(tier, segment, age...) are never components; NPS is not a BHS component so it validates the BHS independently;
CSAT is a component, so CSAT validation uses a leave-CSAT-out BHS.

In [ ]:
BHS_COMPONENTS = {   # name -> (value 0-100, NULL when not observed ; component confidence 0-1 ; description)
    "tx_reliability":    ("CASE WHEN has_transactions = 1 AND n_tx > 0 THEN 100.0 * tx_success_rate END",
                          "least(n_tx / 3.0, 1)", "Share of transactions completed (not failed/reversed)"),
    "spend_level":       ("CASE WHEN has_transactions = 1 AND spend_pctile IS NOT NULL THEN 100.0 * spend_pctile END",
                          "least(n_tx / 3.0, 1)", "Gross spend percentile within month and currency"),
    "spend_trend":       ("CASE WHEN spend_change_vs_base IS NOT NULL THEN 50 + 50 * greatest(-1, least(1, spend_change_vs_base)) END",
                          "spend_base_months / 3.0", "Spend vs own prior-3-month average (50 = stable)"),
    "engagement_level":  ("CASE WHEN has_engagements = 1 THEN 100.0 * eng_pctile END",
                          "least(n_eng / 5.0, 1)", "Engagement events percentile within month"),
    "engagement_quality": ("CASE WHEN positive_engagement_share IS NOT NULL THEN 100.0 * positive_engagement_share END",
                           "least((eng_positive_count + eng_negative_count) / 3.0, 1)", "Positive / (positive + negative) engagement"),
    "engagement_trend":  ("CASE WHEN eng_change_vs_base IS NOT NULL THEN 50 + 50 * greatest(-1, least(1, eng_change_vs_base)) END",
                          "eng_base_months / 3.0", "Engagement vs own prior-3-month average (50 = stable)"),
    "service_experience": ("CASE WHEN int_count > 0 THEN 100.0 * (1 - 0.5 * complaint_rate - 0.5 * unresolved_rate) END",
                           "least(int_count / 2.0, 1)", "Few complaints and unresolved issues"),
    "csat_health":       ("CASE WHEN csat_available = 1 THEN csat_health_score END",
                          "least(csat_response_count / 2.0, 1)", "Mean normalised CSAT (valid ratings)"),
}
if not INCLUDE_CSAT_IN_BHS:
    BHS_COMPONENTS.pop("csat_health")
W = {k: BHS_WEIGHTS[k] for k in BHS_COMPONENTS}
assert abs(sum(BHS_WEIGHTS.values()) - 1) < 1e-9, "BHS_WEIGHTS must sum to 1"

display(Markdown("### 11.1 Components, initial weights and normalisation"))
spec = pd.DataFrame([{"component": k, "initial_weight": W[k], "normalised_range": "0-100", "description": v[2],
                      "value_sql": v[0], "component_confidence_sql": v[1]} for k, v in BHS_COMPONENTS.items()])
save(spec, "s11_1_component_spec", show=12)

In [ ]:
def build_scores():
    comps = ",\n".join(f"({v[0]}) AS c_{k}, CASE WHEN ({v[0]}) IS NOT NULL THEN ({v[1]}) END AS conf_{k}"
                       for k, v in BHS_COMPONENTS.items())
    wobs = " + ".join(f"CASE WHEN c_{k} IS NOT NULL THEN {w} ELSE 0 END" for k, w in W.items())
    num = " + ".join(f"CASE WHEN c_{k} IS NOT NULL THEN {w} * c_{k} ELSE 0 END" for k, w in W.items())
    cnum = " + ".join(f"CASE WHEN c_{k} IS NOT NULL THEN {w} * conf_{k} ELSE 0 END" for k, w in W.items())
    nobs = " + ".join(f"CAST(c_{k} IS NOT NULL AS INT)" for k in W)
    eff = ",\n".join(f"CASE WHEN w_obs > 0 AND c_{k} IS NOT NULL THEN round({w} / w_obs, 4) ELSE 0 END AS weff_{k}" for k, w in W.items())
    least_ = "least(" + ", ".join(f"coalesce(c_{k}, 1e9)" for k in W) + ")"
    weakest = "CASE " + " ".join(f"WHEN coalesce(c_{k}, 1e9) = mn THEN '{k}'" for k in W) + " END"

    global SENTIMENT_SOURCE
    if TEXT_SENTIMENT_PATH.exists():
        SENTIMENT_SOURCE = "Sentiment Agent predictions"
        con.execute(f"""CREATE OR REPLACE TABLE sent_m AS
            SELECT i.customer_id, i.mth, avg(greatest(-1, least(1, CAST(p.text_sentiment_pred AS DOUBLE)))) sentiment_score,
                   count(p.text_sentiment_pred) n_text
            FROM itx i JOIN read_parquet('{TEXT_SENTIMENT_PATH.as_posix()}') p USING (interaction_id)
            WHERE i.in_window GROUP BY ALL""")
    else:
        SENTIMENT_SOURCE = "PROXY: sentiment_score_gt (replace with Sentiment Agent output before production)"
        con.execute("""CREATE OR REPLACE TABLE sent_m AS
            SELECT customer_id, mth, avg(sscore) sentiment_score, count(sscore) n_text FROM itx WHERE in_window GROUP BY ALL""")
    flag("S11", f"Sentiment Score input: {SENTIMENT_SOURCE}")

    con.execute(f"""
    CREATE OR REPLACE TABLE cm_scores AS
    WITH c AS (SELECT f.*, {comps} FROM cm_feat f),
    w AS (SELECT *, {wobs} AS w_obs, {nobs} AS n_components, {least_} AS mn,
                 CASE WHEN ({wobs}) > 0 THEN ({num}) / ({wobs}) END AS bhs_raw,
                 CASE WHEN ({wobs}) > 0 THEN ({cnum}) / ({wobs}) END AS component_confidence
          FROM c),
    s AS (SELECT w.*, {eff},
                 CASE WHEN w_obs >= {MIN_BHS_WEIGHT_COVERAGE} THEN round(bhs_raw, 2) END AS bhs,
                 CASE WHEN w_obs >= {MIN_BHS_WEIGHT_COVERAGE} THEN 'scored' ELSE 'insufficient_evidence' END AS bhs_status,
                 CASE WHEN n_components > 0 THEN {weakest} END AS weakest_component,
                 round(0.40 * w_obs + 0.30 * coalesce(component_confidence, 0)
                       + 0.15 * least(months_observed, 6) / 6.0 + 0.15 * n_sources / 3.0, 3) AS bhs_confidence_score
          FROM w)
    SELECT s.* EXCLUDE (mn),
           CASE WHEN bhs IS NULL THEN 'not_available' WHEN bhs_confidence_score >= 0.70 THEN 'high'
                WHEN bhs_confidence_score >= 0.50 THEN 'medium' ELSE 'low' END AS bhs_confidence_band,
           CASE WHEN bhs IS NULL THEN NULL WHEN bhs < 40 THEN 'at_risk' WHEN bhs < 60 THEN 'watch' ELSE 'healthy' END AS bhs_band,
           -- separate Sentiment score (-1..+1)
           round(t.sentiment_score, 4) AS sentiment_score, coalesce(t.n_text, 0) AS sentiment_n_text,
           CASE WHEN t.n_text > 0 THEN round(least(t.n_text / 3.0, 1), 3) END AS sentiment_confidence,
           -- separate Loyalty score (0-100), NPS-based, NULL when not surveyed
           round(s.nps_loyalty_score, 2) AS loyalty_score, round(s.nps_net_score, 2) AS loyalty_net_score,
           CASE WHEN s.nps_response_count > 0 THEN round(least(s.nps_response_count / 2.0, 1), 3) END AS loyalty_confidence
    FROM s LEFT JOIN sent_m t USING (customer_id, mth)""")
    con.execute(f"COPY cm_scores TO '{(GOLD_DIR / 'customer_month_scores.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")


t0 = time.time()
build_scores()
print(f"cm_scores built in {time.time() - t0:,.0f}s")
display(Markdown("### 11.2 BHS with dynamic weight redistribution + separate Sentiment and Loyalty scores"))
display(q(f"""SELECT customer_id, mth, {', '.join('round(c_' + k + ', 1) c_' + k for k in W)}, n_components, round(w_obs, 2) w_obs,
                     bhs, bhs_band, weakest_component, bhs_confidence_score, bhs_confidence_band,
                     sentiment_score, sentiment_confidence, loyalty_score, loyalty_net_score, loyalty_confidence
              FROM cm_scores WHERE bhs IS NOT NULL LIMIT 8"""))

norm = q(" UNION ALL ".join(f"""SELECT '{k}' component, count(c_{k}) n_observed, round(min(c_{k}), 2) min, round(median(c_{k}), 2) p50,
                                       round(max(c_{k}), 2) max, round(avg(c_{k}), 2) mean FROM cm_scores""" for k in W))
save(norm, "s11_2_component_normalisation_check")
guard_sql = [f"count_if(bhs < 0 OR bhs > 100) out_of_range",
             f"count_if(bhs IS NOT NULL AND abs({' + '.join('weff_' + k for k in W)} - 1) > 1e-3) eff_weights_not_1",
             f"count_if(bhs IS NOT NULL AND w_obs < {MIN_BHS_WEIGHT_COVERAGE}) below_min_coverage_scored"]
if "csat_health" in W:
    guard_sql.append("count_if(csat_available = 0 AND weff_csat_health > 0) csat_weight_without_csat")
chk = save(q(f"SELECT {', '.join(guard_sql)} FROM cm_scores"), "s11_2_redistribution_guard_tests")
flag("S11", "BHS guard tests " + ("PASS" if int(chk.values.sum()) == 0 else "FAIL - see s11_2_redistribution_guard_tests.csv"))

In [ ]:
display(Markdown("### 11.3 Score coverage and component-level coverage & confidence"))
cov = save(q("""SELECT mth, count(*) active_cm, round(100.0*count(bhs)/count(*), 2) pct_bhs_scored,
                       round(avg(w_obs), 3) avg_weight_coverage, round(avg(n_components), 2) avg_components,
                       round(100.0*count(sentiment_score)/count(*), 2) pct_sentiment_scored,
                       round(100.0*count(loyalty_score)/count(*), 2) pct_loyalty_scored
                FROM cm_scores GROUP BY 1 ORDER BY 1"""), "s11_3_score_coverage_by_month", show=12)
save(q("""SELECT n_components, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct,
                 round(100.0*count(bhs)/count(*), 2) pct_scored FROM cm_scores GROUP BY 1 ORDER BY 1"""), "s11_3_components_observed")
comp_cov = save(q(" UNION ALL ".join(
    f"""SELECT '{k}' component, {W[k]} initial_weight, round(100.0*count(c_{k})/count(*), 2) pct_observed,
               round(avg(conf_{k}), 3) avg_component_confidence,
               round(avg(weff_{k}) FILTER (WHERE bhs IS NOT NULL), 3) avg_effective_weight
        FROM cm_scores""" for k in W)), "s11_3_component_coverage_confidence", show=12)
flag("S11", f"BHS published for {cov.pct_bhs_scored.mean():.1f}% of active customer-months on average; "
            f"lowest-coverage component: {comp_cov.sort_values('pct_observed').component.iloc[0]} "
            f"({comp_cov.pct_observed.min():.1f}% observed)")

display(Markdown("### 11.4 Confidence"))
save(q("""SELECT bhs_confidence_band, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct,
                 round(avg(w_obs), 3) avg_weight_coverage, round(avg(component_confidence), 3) avg_component_conf,
                 round(avg(n_sources), 2) avg_sources, round(avg(months_observed), 2) avg_months_observed
          FROM cm_scores GROUP BY 1 ORDER BY 2 DESC"""), "s11_4_confidence_bands")
save(q("""SELECT bhs_band, count(*) n, round(avg(bhs), 2) avg_bhs, mode(weakest_component) most_common_weakest_component
          FROM cm_scores WHERE bhs IS NOT NULL GROUP BY 1 ORDER BY 3"""), "s11_4_bhs_bands")

h = q("SELECT bhs, bhs_confidence_score FROM cm_scores WHERE bhs IS NOT NULL USING SAMPLE reservoir(200000 ROWS) REPEATABLE (1)")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(h.bhs, bins=50); ax[0].set_title("BHS distribution (200k sample)")
ax[1].hist(h.bhs_confidence_score, bins=40, color="#FFC000"); ax[1].set_title("BHS confidence score")
plt.tight_layout(); plt.savefig(OUT_DIR / "s11_bhs_distribution.png", dpi=120); finish_plot()
del h

In [ ]:
display(Markdown("### 11.5 Weight sensitivity"))
COMP_COLS = [f"c_{k}" for k in W]
S = q(f"""SELECT {', '.join(COMP_COLS)}, bhs FROM cm_scores WHERE bhs IS NOT NULL
          USING SAMPLE reservoir({SENS_SAMPLE} ROWS) REPEATABLE ({SEED})""")
C = S[COMP_COLS].to_numpy(dtype=float)
OBS = ~np.isnan(C)


def bhs_np(weights, comps=None, obs=None):
    comps = C if comps is None else comps
    obs = OBS if obs is None else obs
    w = np.array([weights.get(k, 0.0) for k in W], dtype=float)
    w = w / w.sum() if w.sum() > 0 else w
    wo = obs * w
    ws = wo.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        s = (np.nan_to_num(comps) * wo).sum(1) / ws      # nan_to_num only where weight is 0 -> excluded, not imputed
    s[ws <= 0] = np.nan
    return s


base = bhs_np(W)
assert np.nanmax(np.abs(base - S.bhs.values)) < 0.01, "Python BHS != SQL BHS"
base_risk = base <= np.nanquantile(base, 0.10)


def compare(name, w):
    s = bhs_np(w)
    m = ~np.isnan(s) & ~np.isnan(base)
    risk = s <= np.nanquantile(s, 0.10)
    return {"scenario": name, "spearman_vs_base": spearmanr(base[m], s[m])[0],
            "at_risk_decile_overlap": (risk & base_risk).sum() / max(base_risk.sum(), 1),
            "mean_abs_change": np.nanmean(np.abs(s - base))}


rows = []
for k in W:
    for f in (0.5, 1.5):
        rows.append(compare(f"{k} x{f}", {**W, k: W[k] * f}))
    rows.append(compare(f"drop {k}", {**W, k: 0.0}))
rows.append(compare("equal weights", {k: 1.0 for k in W}))
rng = np.random.default_rng(SEED)
rand = pd.DataFrame([compare(f"random_{i}", dict(zip(W, rng.dirichlet(np.array(list(W.values())) * 50))))
                     for i in range(SENS_RANDOM_RUNS)])
sens = save(pd.DataFrame(rows).sort_values("spearman_vs_base"), "s11_5_weight_sensitivity", show=30)
save(rand[["spearman_vs_base", "at_risk_decile_overlap", "mean_abs_change"]].describe().T.reset_index(), "s11_5_random_weight_stability")
worst = sens.sort_values("at_risk_decile_overlap").iloc[0]
flag("S11", f"Sensitivity: most influential change = '{worst.scenario}' (at-risk decile overlap {worst.at_risk_decile_overlap:.1%}, "
            f"rank corr {worst.spearman_vs_base:.3f}); random weight perturbations keep median overlap "
            f"{rand.at_risk_decile_overlap.median():.1%}")
del S, C, OBS
gc.collect()

In [ ]:
display(Markdown("### 11.6 Independent validation vs CSAT (leave-CSAT-out) and NPS (not in BHS)"))
V = q(f"""SELECT {', '.join(COMP_COLS)}, bhs, avg_csat_rating, csat_available, nps_available, avg_nps_rating,
                 nps_detractor_count, nps_promoter_count, loyalty_score, sentiment_score
          FROM cm_scores WHERE bhs IS NOT NULL AND (csat_available = 1 OR nps_available = 1)""")
Cv = V[COMP_COLS].to_numpy(dtype=float)
V["bhs_wo_csat"] = bhs_np({**W, "csat_health": 0.0}, Cv, ~np.isnan(Cv)) if "csat_health" in W else V.bhs
del Cv
c = V[V.csat_available == 1]
n_ = V[V.nps_available == 1]
det = (n_.nps_detractor_count > n_.nps_promoter_count).astype(int)
val = save(pd.DataFrame([
    {"check": "Spearman BHS(leave-CSAT-out) vs avg CSAT", "value": spearmanr(c.bhs_wo_csat, c.avg_csat_rating, nan_policy="omit")[0], "n": len(c)},
    {"check": "AUC low CSAT (<=2) from low BHS(leave-CSAT-out)", "value": auc_safe((c.avg_csat_rating <= 2).astype(int), -c.bhs_wo_csat), "n": len(c)},
    {"check": "Spearman BHS vs avg NPS", "value": spearmanr(n_.bhs, n_.avg_nps_rating, nan_policy="omit")[0], "n": len(n_)},
    {"check": "AUC net-detractor month from low BHS", "value": auc_safe(det, -n_.bhs), "n": len(n_)},
    {"check": "Spearman Loyalty score vs Sentiment score (separate scores)",
     "value": spearmanr(n_.loyalty_score, n_.sentiment_score, nan_policy="omit")[0], "n": len(n_)},
    {"check": "Spearman BHS vs Sentiment score", "value": spearmanr(V.bhs, V.sentiment_score, nan_policy="omit")[0], "n": len(V)},
]), "s11_6_validation_csat_nps")
save(c.assign(csat_round=c.avg_csat_rating.round()).groupby("csat_round")
      .agg(n=("bhs", "size"), avg_bhs_wo_csat=("bhs_wo_csat", "mean")).reset_index(), "s11_6_bhs_by_csat")
save(n_.assign(cat=np.select([n_.avg_nps_rating <= 6, n_.avg_nps_rating <= 8], ["detractor", "passive"], "promoter"))
       .groupby("cat").agg(n=("bhs", "size"), avg_bhs=("bhs", "mean")).reset_index(), "s11_6_bhs_by_nps_category")
flag("S11", f"Independent validation: BHS vs NPS Spearman {val.value.iloc[2]:.3f}, detractor AUC {val.value.iloc[3]:.3f}; "
            f"leave-CSAT-out BHS vs CSAT Spearman {val.value.iloc[0]:.3f}")
del V, c, n_, det
gc.collect()

In [ ]:
display(Markdown(f"### 11.7 Out-of-time validation (calibration months < {CAL_END} <= validation months) and weight suggestion"))
con.execute(f"""
CREATE OR REPLACE TABLE bhs_oot AS
SELECT s.mth, s.bhs, {', '.join('s.' + c for c in COMP_COLS)},
       count(*) OVER fut AS active_months_next,                                   -- cm_scores holds active months only
       first_value(s.latest_nps_rating IGNORE NULLS) OVER fut AS next_nps,
       CASE WHEN s.mth < DATE '{CAL_END}' THEN 'in_time' ELSE 'out_of_time' END period
FROM cm_scores s
WINDOW fut AS (PARTITION BY s.customer_id ORDER BY s.mth
               RANGE BETWEEN INTERVAL 1 MONTH FOLLOWING AND INTERVAL {OUTCOME_HORIZON} MONTH FOLLOWING)""")
O = q(f"""SELECT * FROM (SELECT * FROM bhs_oot WHERE bhs IS NOT NULL
                         AND mth <= DATE '{WINDOW_END}' - INTERVAL {OUTCOME_HORIZON + 1} MONTH)
          USING SAMPLE reservoir({OOT_SAMPLE} ROWS) REPEATABLE ({SEED})""")
con.execute("DROP TABLE IF EXISTS bhs_oot")
O[COMP_COLS + ["bhs"]] = O[COMP_COLS + ["bhs"]].astype("float32")
O["y_inactive"] = (O.active_months_next == 0).astype(int)
O["y_future_detractor"] = np.where(O.next_nps.isna(), np.nan, (O.next_nps <= 6).astype(float))

rows = []
for p, g in O.groupby("period"):
    fd = g[g.y_future_detractor.notna()]
    rows.append({"period": p, "months": f"{g.mth.min():%Y-%m}..{g.mth.max():%Y-%m}", "n": len(g),
                 "inactive_rate": g.y_inactive.mean(), "auc_inactivity_low_bhs": auc_safe(g.y_inactive, -g.bhs),
                 "n_future_nps": len(fd), "auc_future_detractor_low_bhs": auc_safe(fd.y_future_detractor.astype(int), -fd.bhs)})
oot = save(pd.DataFrame(rows), "s11_7_out_of_time_validation")

T = O[O.period == "out_of_time"].copy()
T["decile"] = pd.qcut(T.bhs.rank(method="first"), 10, labels=False) + 1
dec = save(T.groupby("decile").agg(n=("bhs", "size"), avg_bhs=("bhs", "mean"), inactive_rate=("y_inactive", "mean"),
                                   future_detractor_rate=("y_future_detractor", "mean")).reset_index(), "s11_7_oot_deciles")

I = O[O.period == "in_time"]
uni = []
for k in W:   # univariate AUC of each component on IN-TIME data (observed rows only, no imputation)
    m = I[f"c_{k}"].notna()
    uni.append({"component": k, "n_observed": int(m.sum()),
                "auc_inactivity_low_value": auc_safe(I.loc[m, "y_inactive"], -I.loc[m, f"c_{k}"])})
uni = pd.DataFrame(uni)
lift = (uni.auc_inactivity_low_value - 0.5).clip(lower=0.0).fillna(0) + 0.01     # small floor keeps every component
uni["suggested_weight"] = (lift / lift.sum()).round(3)
uni["initial_weight"] = uni.component.map(W)
save(uni, "s11_7_component_auc_and_suggested_weights", show=12)
Co = T[COMP_COLS].to_numpy(dtype=float); Oo = ~np.isnan(Co)
sugg = dict(zip(uni.component, uni.suggested_weight))
cmp_ = save(pd.DataFrame([
    {"weights": "initial", "oot_auc_inactivity": auc_safe(T.y_inactive, -bhs_np(W, Co, Oo))},
    {"weights": "suggested (in-time AUC)", "oot_auc_inactivity": auc_safe(T.y_inactive, -bhs_np(sugg, Co, Oo))},
    {"weights": "equal", "oot_auc_inactivity": auc_safe(T.y_inactive, -bhs_np({k: 1 for k in W}, Co, Oo))}]),
    "s11_7_weights_oot_comparison")
a = oot.set_index("period").auc_inactivity_low_bhs
flag("S11", f"Out-of-time: inactivity AUC {a.get('out_of_time'):.3f} vs in-time {a.get('in_time'):.3f}; suggested weights OOT AUC "
            f"{cmp_.oot_auc_inactivity.iloc[1]:.3f} vs initial {cmp_.oot_auc_inactivity.iloc[0]:.3f}. "
            "Review suggested weights with the business before adopting (BHS_WEIGHTS in 1A)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(dec.decile, dec.inactive_rate * 100, marker="o"); ax[0].set_title(f"OOT: inactive next {OUTCOME_HORIZON}m by BHS decile")
ax[0].set_xlabel("BHS decile (1 = lowest)"); ax[0].set_ylabel("%")
ax[1].plot(dec.decile, dec.future_detractor_rate * 100, marker="o", color="#C00000"); ax[1].set_title("OOT: future detractor rate by BHS decile")
ax[1].set_xlabel("BHS decile"); ax[1].set_ylabel("%")
for ax_ in ax:
    ax_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "s11_7_oot_deciles.png", dpi=120); finish_plot()

del O, T, I, Co, Oo     # release memory before Step 12
gc.collect()

## Step 12 - Cohort selection scorecard (former Step 11 - logic unchanged)

In [ ]:
COHORT_DIMS = ["customer_segment", "loyalty_tier", "region", "preferred_channel", "tenure_band"]
WEIGHTS = {"value": 0.25, "coverage": 0.20, "labels": 0.15, "sample": 0.15,
           "history": 0.10, "stability": 0.10, "homogeneity": 0.05}


def step12_cohort():
    header("STEP 12 - Cohort selection scorecard")
    segment_view()
    frames = []
    dims = [("ALL", "'ALL'")] + [(d, f"CAST({d} AS VARCHAR)") for d in COHORT_DIMS]
    for dim, expr in dims:
        frames.append(q(f"""
            SELECT '{dim}' dimension, {expr} segment,
                   count(DISTINCT customer_id) FILTER (WHERE {CORE_COND}) n_core_customers,
                   count_if({CORE_COND}) n_core_customer_months,
                   round(100.0*count_if({CORE_COND})/nullif(count_if({HISTORY_COND}), 0), 2) coverage_pct,
                   round(avg(active_months_to_date) FILTER (WHERE {CORE_COND}), 2) avg_active_months_history,
                   round(100.0*count_if({CORE_COND} AND (n_csat > 0 OR n_nps > 0))/nullif(count_if({CORE_COND}),0), 2) survey_label_pct,
                   round(100.0*count_if({CORE_COND} AND has_int)/nullif(count_if({CORE_COND}),0), 2) interaction_label_pct,
                   round(avg(spend_pctile) FILTER (WHERE {CORE_COND}), 4) avg_spend_pctile,
                   round(stddev(spend_pctile) FILTER (WHERE {CORE_COND}), 4) std_spend_pctile,
                   round(100.0*count_if({CORE_COND} AND (n_complaint > 0 OR n_negative > 0))/nullif(count_if({CORE_COND}),0), 2) risk_signal_pct
            FROM cm_seg GROUP BY ALL"""))
        monthly = q(f"""SELECT {expr} segment, mth, count_if({CORE_COND}) n FROM cm_seg
                        WHERE months_observed >= {MIN_HISTORY_MONTHS} GROUP BY ALL ORDER BY 2""")
        monthly = monthly[monthly.n > 0]
        stab = (monthly.sort_values("mth").groupby("segment").n
                .apply(lambda s: (s.pct_change().abs() * 100).max()).rename("max_mom_swing_pct").reset_index())
        frames[-1] = frames[-1].merge(stab, on="segment", how="left")
    sc = pd.concat(frames, ignore_index=True)

    cand = sc[sc.dimension != "ALL"].copy()
    cand["passes_sample_gate"] = cand.n_core_customers >= MIN_COHORT_CUSTOMERS
    rank = lambda s, asc=True: s.rank(pct=True, ascending=asc)
    # value = spend level (within-currency percentile) + how much risk there is to act on
    cand["s_value"] = rank(cand.avg_spend_pctile) * 0.5 + rank(cand.risk_signal_pct) * 0.5
    cand["s_coverage"] = rank(cand.coverage_pct)
    cand["s_labels"] = rank(cand.survey_label_pct) * 0.5 + rank(cand.interaction_label_pct) * 0.5
    cand["s_sample"] = rank(cand.n_core_customers)
    cand["s_history"] = rank(cand.avg_active_months_history)
    cand["s_stability"] = rank(cand.max_mom_swing_pct, asc=False)
    cand["s_homogeneity"] = rank(cand.std_spend_pctile, asc=False)
    cand["cohort_score"] = sum(cand[f"s_{k}"] * w for k, w in WEIGHTS.items()).round(3)
    cand.loc[~cand.passes_sample_gate, "cohort_score"] = 0
    cand = cand.sort_values("cohort_score", ascending=False)

    save(sc[sc.dimension == "ALL"], "s12_core_cohort_overall")
    save(cand, "s12_cohort_scorecard", show=15)
    top = cand.iloc[0]
    flag("S12", f"Top candidate cohort: {top.dimension} = {top.segment} (score {top.cohort_score}, "
                f"{top.n_core_customers:,} core customers, coverage {top.coverage_pct}%). "
                "Confirm with business value before final selection.")


run_step(step12_cohort)

## Summary - auto-flagged findings

In [ ]:
def write_summary(chosen):
    all_steps = sorted(chosen) == sorted(STEPS)
    name = "eda_summary.md" if all_steps else f"eda_summary_steps_{'_'.join(map(str, chosen))}.md"
    lines = ["# EDA summary - Customer Health & Sentiment Scoring", "",
             f"Data window: {WINDOW_START} to {WINDOW_END} (exclusive) | grain: customer_id x month", "",
             "## Auto-flagged findings", ""]
    for step, msg in FINDINGS:
        lines.append(f"- **{step}** {msg}")
    lines += ["", "## Output files", ""] + [f"- {p.name}" for p in sorted(OUT_DIR.glob("*.csv"))]
    (OUT_DIR / name).write_text("\n".join(lines), encoding="utf-8")
    print(f"\nSummary written to {OUT_DIR / name} ({len(FINDINGS)} findings)")



STEPS = {1: step1_structure, 2: step2_quality, 3: step3_univariate, 4: step4_time, 5: step5_coverage,
         6: step6_lifecycle, 7: step7_interactions, 8: step8_relationships, 9: step9_segments,
         10: step10_leakage_eligibility, 11: "bhs_calibration (section 11)", 12: step12_cohort}
write_summary(list(STEPS))
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(FINDINGS, columns=["step", "finding"]))

In [ ]:
con.close()   # release the DuckDB file lock